# v3 on Kaggle T4: TensorRT-LLM FP16 vs vLLM, Qwen2.5-1.5B-Instruct

Item 12 of `docs/PLAN.md` (`industrial-llm-platform`). One-click **Run all** on **GPU T4 x2** with internet on.
Pushed with `kaggle kernels push -p notebooks/kaggle-trtllm -t 10800` (3 h hard cap); the notebook's own budget
(`BUDGET_S`) skips steps that would not finish. Estimate: about 1.6 h (vLLM control about 0.6 h, then TensorRT-LLM). Uses one T4.

**Rerun scope (`RUN_VLLM_CONTROL = True`).** The vLLM control succeeded on 2026-10-10
(`benchmark/results/raw/kaggle-20261010-v3d/`, 9 runs and GSM8K) and runs again here, so TensorRT-LLM has a
same-session comparison. There the TensorRT-LLM venv installed but
`import tensorrt_llm` failed: datasets 2.14.4 subclasses `pyarrow.PyExtensionType`, which pyarrow 21 removed, and
the lock had pyarrow 25.0.1. The lock now pins pyarrow 20.0.0 and pynvml 11.5.3 (pynvml 13 has no `__version__`,
which `tensorrt_llm/profiler.py` reads at import: the next failure, found locally). `import tensorrt_llm` succeeded
in a local Ubuntu 24.04 container with CUDA driver and NVML stub libraries. The install step now also logs `uv pip check`. With the control on, `benchmark/report.py`
compares the TensorRT-LLM rows with this session's vLLM rows only.
The third run (v3f) got past install and import, then the checkpoint save failed because TensorRT-LLM 0.14's
`save_checkpoint` does not create its output dir; `inference/trtllm/build_engine.py` now creates it, and the
conversion of the real model ran to completion on CPU in a local container (the engine build needs the GPU).
The fourth run (v3h) failed in trtllm-build: unfused context attention needed 21.98 GB of workspace for batch 32 and
input 3072. Two engines now: `bench` (batch 32, input 1024) for loadgen, `gsm8k` (batch 8, input 2048) for GSM8K,
about 2.55 GB of workspace each (calculation in `inference/trtllm/build_engine.py`).

**Why TensorRT-LLM 0.14.0.** It is the last release whose support matrix lists Turing (SM75: FP32, FP16, INT8 (no
SmoothQuant on SM75), INT4). 0.15.0 drops Turing from the matrix and 0.16.0 stops compiling SM70 kernels (which
Turing runs). The 0.14.0 wheel is Python 3.10 only, so this notebook builds a 3.10 venv for it. FP16 only on T4.

Steps: Open MPI 4.1.6 from Ubuntu (pinned), venvs, model download (pinned revision), engine build
(`inference/trtllm/build_engine.py`: untie embeddings, FP16 checkpoint, `trtllm-build --context_fmha disable`, since
fused context attention has no SM75 kernels), OpenAI-compatible server (`inference/trtllm/serve.py` around 0.14.0's
`examples/apps/openai_server.py`, sha256-pinned, one documented stream patch, plus `/metrics`), then loadgen at
concurrency 1, 8, 32, 3 repeats, 128-token fixed decode, and GSM8K (250 items) if time allows.

**Same-session vLLM control.** vLLM 0.31.0, same model, FP16, same loadgen cells and GSM8K, with
`--generation-config vllm` (no repetition_penalty), so the TensorRT-LLM vs vLLM comparison is same hardware, same
session, same sampling. `benchmark/report.py` compares TensorRT-LLM only with vLLM rows of the same session, and
generates the like-for-like check (prompts sha256, max_tokens, actual token counts, truncated streams) from the raw
files. Run v2 took 978 s for this cell.

**Not run: Triton's TensorRT-LLM backend.** It ships only in the NGC `tritonserver:<ver>-trtllm-python-py3` image,
which needs Docker; Kaggle has none. Recorded as `NOT_RUN-triton-tensorrtllm-backend.txt`.

If TensorRT-LLM cannot be installed, built or served, the failing step writes `FAILED-<step>.txt` with the
traceback and every later step writes `NOT_RUN-<step>.txt` with the reason. Preflight stops at once without GPU or
internet (`FAILED-preflight.txt`).


In [ ]:
# Pinned configuration. Everything else is in the embedded lock files.
GIT_COMMIT = "693b44b4446c00948085c81325bbeb24b5897776"  # repo commit the embedded code came from
UV_VERSION = "0.11.32"
PYTHON_VERSION = "3.12.13"  # client venv (loadgen, lm-eval)
TRTLLM_PYTHON = "3.10.20"  # TensorRT-LLM 0.14.0 ships cp310 wheels only
TRTLLM_VERSION = "0.14.0"
OPENMPI_APT = "4.1.6-7ubuntu2"  # Ubuntu 24.04 (noble) universe
OPENAI_SERVER_URL = (
    "https://raw.githubusercontent.com/NVIDIA/TensorRT-LLM/"
    "b0880169d0fb8cd0363049d91aa548e58a41be07/examples/apps/openai_server.py"
)  # tag v0.14.0
OPENAI_SERVER_SHA256 = (
    "8ae2d1d892792bada31424bd8295dc87a5e1a902c2f723427e424a5a1b3f9f1f"
)
MODELS = {
    "1.5b-fp16": (
        "Qwen/Qwen2.5-1.5B-Instruct",
        "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
    )
}
REPEATS = 3
GSM8K_LIMIT = 250
PORT = 8000
KV_FRACTION = 0.90  # of the GPU memory free after the engine loads (vLLM's 0.90 is of the whole card)
WORK = "/kaggle/working"
RESULTS = f"{WORK}/results"
BENCH = f"{WORK}/bench"
VENVS = "/tmp/venvs"
TRT_WORK = "/tmp/trtllm"  # HF snapshot, checkpoint and engine (not saved as output)
BUDGET_S = 10200  # push -t 10800; leaves 10 min for the summary
CELL = "trtllm-t4-1.5b-fp16"
VLLM_VERSION = "0.31.0"  # same-session control, as run v2 plus --generation-config vllm
MAX_MODEL_LEN = 4096
GPU_MEM_UTIL = 0.90
VLLM_CELL = "vllm-t4-1.5b-fp16"
# Re-run the vLLM control so TensorRT-LLM has same-session vLLM rows (v3d's control is from another session).
RUN_VLLM_CONTROL = True
CONTROL_EVIDENCE = (
    "benchmark/results/raw/kaggle-20261010-v3d/ (notebook built at 80905ea)"
)
PINS = {
    "tensorrt_llm": TRTLLM_VERSION,
    "tensorrt": "10.4.0",
    "torch": "2.4.0+cu124",
    "python": TRTLLM_PYTHON,
    "openmpi": OPENMPI_APT,
    "vllm_version": VLLM_VERSION,
}
PREFLIGHT_URLS = (
    "https://pypi.org/simple/",
    "https://pypi.nvidia.com/tensorrt-llm/",
    "https://github.com",
    "https://huggingface.co",
    OPENAI_SERVER_URL,
)

In [ ]:
# Embedded copies of the repo files (generated by notebooks/build_notebook.py). sha256 checked on write.
import hashlib
import json
import os
import pathlib

FILES = json.loads(
    '{"benchmark/loadgen/run.py": ["9c1bd8b99b72a49a57c86f9ff3739215d27db837744711a4b70f5b1bd256e04f", "\\"\\"\\"Closed-loop streaming load generator for any OpenAI-compatible chat endpoint (llama.cpp, vLLM).\\n\\nWrites one raw JSON per (concurrency, repeat) with full config, environment, per-request records and summary.\\nExample:\\n  python benchmark/loadgen/run.py --base-url http://127.0.0.1:8080/v1 --model qwen2.5-1.5b-instruct-q4_k_m \\\\\\n    --engine llama.cpp --engine-version b11459 --precision gguf-q4_k_m --label llamacpp-cpu-1.5b-q4km \\\\\\n    --concurrency 1 4 8 --repeats 3 --out-dir benchmark/results/raw\\n\\"\\"\\"\\n\\nimport argparse\\nimport asyncio\\nimport datetime as dt\\nimport hashlib\\nimport json\\nimport os\\nimport platform\\nimport shutil\\nimport subprocess\\nimport sys\\nimport tempfile\\nimport time\\nfrom pathlib import Path\\n\\nimport httpx\\n\\nsys.path.insert(0, str(Path(__file__).parent))\\nfrom metrics import decode_checks, summarize\\n\\nHERE = Path(__file__).parent\\n\\n\\ndef utcnow():\\n    return dt.datetime.now(dt.UTC).isoformat(timespec=\\"seconds\\")\\n\\n\\ndef sh(*cmd):\\n    try:\\n        return subprocess.run(\\n            cmd, capture_output=True, text=True, timeout=30, check=False\\n        ).stdout.strip()\\n    except (OSError, subprocess.SubprocessError):\\n        return None\\n\\n\\ndef cpu_model():\\n    if Path(\\"/proc/cpuinfo\\").exists():\\n        for line in Path(\\"/proc/cpuinfo\\").read_text().splitlines():\\n            if line.startswith(\\"model name\\"):\\n                return line.split(\\":\\", 1)[1].strip()\\n    if sys.platform == \\"win32\\":\\n        import winreg\\n\\n        key = winreg.OpenKey(\\n            winreg.HKEY_LOCAL_MACHINE, r\\"HARDWARE\\\\DESCRIPTION\\\\System\\\\CentralProcessor\\\\0\\"\\n        )\\n        return winreg.QueryValueEx(key, \\"ProcessorNameString\\")[0].strip()\\n    return platform.processor()\\n\\n\\ndef git_commit():\\n    commit = sh(\\"git\\", \\"-C\\", str(HERE), \\"rev-parse\\", \\"HEAD\\")\\n    if not commit:\\n        return None\\n    return commit + (\\n        \\"-dirty\\" if sh(\\"git\\", \\"-C\\", str(HERE), \\"status\\", \\"--porcelain\\") else \\"\\"\\n    )\\n\\n\\nasync def one_request(client, args, prompt, warmup, t0):\\n    body = {\\n        \\"model\\": args.model,\\n        \\"messages\\": [{\\"role\\": \\"user\\", \\"content\\": prompt[\\"prompt\\"]}],\\n        \\"max_tokens\\": args.max_tokens,\\n        \\"temperature\\": 0,\\n        \\"seed\\": args.seed,\\n        \\"stream\\": True,\\n        \\"stream_options\\": {\\"include_usage\\": True},\\n    }\\n    if args.ignore_eos:\\n        body[\\"ignore_eos\\"] = (\\n            True  # llama.cpp and vLLM both accept it; fixes output length\\n        )\\n    rec = {\\n        \\"prompt_id\\": prompt[\\"id\\"],\\n        \\"kind\\": prompt[\\"kind\\"],\\n        \\"warmup\\": warmup,\\n        \\"token_times\\": [],\\n        \\"error\\": None,\\n    }\\n    chunks, usage = 0, None\\n    rec[\\"start\\"] = round(time.perf_counter() - t0, 6)\\n    try:\\n        async with client.stream(\\n            \\"POST\\", f\\"{args.base_url}/chat/completions\\", json=body\\n        ) as resp:\\n            if resp.status_code != 200:\\n                rec[\\"error\\"] = (\\n                    f\\"HTTP {resp.status_code}: {(await resp.aread()).decode(errors=\'replace\')[:300]}\\"\\n                )\\n            else:\\n                async for line in resp.aiter_lines():\\n                    if not line.startswith(\\"data: \\"):\\n                        continue\\n                    data = line[6:].strip()\\n                    if data == \\"[DONE]\\":\\n                        break\\n                    msg = json.loads(data)\\n                    if msg.get(\\"usage\\"):\\n                        usage = msg[\\"usage\\"]\\n                    for ch in msg.get(\\"choices\\") or []:\\n                        if (ch.get(\\"delta\\") or {}).get(\\"content\\"):\\n                            rec[\\"token_times\\"].append(\\n                                round(time.perf_counter() - t0, 6)\\n                            )\\n                            chunks += 1\\n    except Exception as e:  # noqa: BLE001 recorded, never raised: a failed request is data\\n        rec[\\"error\\"] = f\\"{type(e).__name__}: {e}\\"\\n    rec[\\"end\\"] = round(time.perf_counter() - t0, 6)\\n    rec[\\"chunks\\"] = chunks\\n    rec[\\"prompt_tokens\\"] = (usage or {}).get(\\"prompt_tokens\\")\\n    rec[\\"output_tokens\\"] = (usage or {}).get(\\"completion_tokens\\", chunks)\\n    rec[\\"token_source\\"] = (\\n        \\"usage\\" if usage and \\"completion_tokens\\" in usage else \\"chunks\\"\\n    )\\n    if not rec[\\"error\\"] and chunks == 0:\\n        rec[\\"error\\"] = \\"no tokens streamed\\"\\n    return rec\\n\\n\\nasync def closed_loop(client, args, prompts, n, concurrency, warmup, t0):\\n    \\"\\"\\"concurrency workers, each sends its next request as soon as its previous one ends.\\"\\"\\"\\n    it = iter(range(n))\\n    out = []\\n\\n    async def worker():\\n        for i in it:\\n            out.append(\\n                await one_request(client, args, prompts[i % len(prompts)], warmup, t0)\\n            )\\n\\n    await asyncio.gather(*(worker() for _ in range(concurrency)))\\n    return out\\n\\n\\nclass GpuSampler:\\n    \\"\\"\\"nvidia-smi every 500 ms in the background, written to a file (no pipe to drain); no-op without nvidia-smi.\\"\\"\\"\\n\\n    # clocks_event_reasons.active: bitmask (0x4 SW power cap, 0x20 SW thermal, 0x40 HW thermal); needs driver >= 535.\\n    QUERY = (\\n        \\"timestamp,index,memory.used,memory.total,utilization.gpu,clocks.sm,temperature.gpu,power.draw,\\"\\n        \\"clocks.mem,clocks_event_reasons.active\\"\\n    )\\n    KEYS = (\\n        \\"timestamp\\",\\n        \\"gpu\\",\\n        \\"memory_used_mib\\",\\n        \\"memory_total_mib\\",\\n        \\"util_gpu_pct\\",\\n        \\"clock_sm_mhz\\",\\n        \\"temp_c\\",\\n        \\"power_w\\",\\n        \\"clock_mem_mhz\\",\\n        \\"clocks_event_reasons\\",\\n    )\\n\\n    def __init__(self):\\n        self.proc = None\\n        if shutil.which(\\"nvidia-smi\\"):\\n            fd, self.path = tempfile.mkstemp(suffix=\\".csv\\")\\n            os.close(fd)\\n            ids = os.environ.get(\\n                \\"CUDA_VISIBLE_DEVICES\\"\\n            )  # sample only the GPUs the server uses\\n            cmd = [\\n                \\"nvidia-smi\\",\\n                f\\"--query-gpu={self.QUERY}\\",\\n                \\"--format=csv,noheader,nounits\\",\\n                \\"-lms\\",\\n                \\"500\\",\\n            ]\\n            cmd += [\\"-i\\", ids] if ids else []\\n            self.proc = subprocess.Popen(\\n                [*cmd, \\"-f\\", self.path],\\n                stdout=subprocess.DEVNULL,\\n                stderr=subprocess.DEVNULL,\\n            )\\n\\n    def stop(self):\\n        if not self.proc:\\n            return None\\n        self.proc.terminate()\\n        try:\\n            self.proc.wait(timeout=10)\\n        except subprocess.TimeoutExpired:\\n            self.proc.kill()\\n        rows = Path(self.path).read_text().splitlines()\\n        Path(self.path).unlink()\\n        return [\\n            dict(zip(self.KEYS, [c.strip() for c in r.split(\\",\\")], strict=False))\\n            for r in rows\\n            if r.strip()\\n        ]\\n\\n\\nclass DcgmSampler:\\n    \\"\\"\\"dcgm_sampler.py in the background (--dcgm-fields), same lifecycle as GpuSampler; no-op without fields.\\n    GPU ids are taken from CUDA_VISIBLE_DEVICES, assuming CUDA and DCGM number the GPUs the same way.\\"\\"\\"\\n\\n    def __init__(self, fields, host):\\n        self.proc = None\\n        if not fields:\\n            return\\n        fd, self.path = tempfile.mkstemp(suffix=\\".csv\\")\\n        os.close(fd)\\n        fd, self.err = tempfile.mkstemp(suffix=\\".err\\")\\n        os.close(fd)\\n        ids = os.environ.get(\\"CUDA_VISIBLE_DEVICES\\")\\n        cmd = [\\n            sys.executable,\\n            str(HERE / \\"dcgm_sampler.py\\"),\\n            \\"--fields\\",\\n            fields,\\n            \\"--host\\",\\n            host,\\n        ]\\n        cmd += [\\"--out\\", self.path] + ([\\"--gpus\\", ids] if ids else [])\\n        with open(self.err, \\"w\\") as err:\\n            self.proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=err)\\n\\n    def stop(self):\\n        \\"\\"\\"(samples, error text or None); (None, None) when off.\\"\\"\\"\\n        if not self.proc:\\n            return None, None\\n        self.proc.terminate()\\n        try:\\n            self.proc.wait(timeout=10)\\n        except subprocess.TimeoutExpired:\\n            self.proc.kill()\\n        lines = Path(self.path).read_text().splitlines()\\n        keys = lines[0].split(\\",\\") if lines else []  # header written by dcgm_sampler.py\\n        rows = lines[1:]\\n        err = Path(self.err).read_text().strip() or None\\n        Path(self.path).unlink()\\n        Path(self.err).unlink()\\n        return [\\n            dict(zip(keys, r.split(\\",\\"), strict=False)) for r in rows if r.strip()\\n        ], err\\n\\n\\nasync def docker_stats():\\n    if not shutil.which(\\"docker\\"):\\n        return None\\n    try:\\n        p = await asyncio.create_subprocess_exec(\\n            \\"docker\\",\\n            \\"stats\\",\\n            \\"--no-stream\\",\\n            \\"--format\\",\\n            \\"{{json .}}\\",\\n            stdout=asyncio.subprocess.PIPE,\\n            stderr=asyncio.subprocess.DEVNULL,\\n        )\\n        out, _ = await asyncio.wait_for(p.communicate(), 30)\\n        return [json.loads(line) for line in out.decode().splitlines() if line.strip()]\\n    except (TimeoutError, OSError, json.JSONDecodeError):\\n        return None\\n\\n\\nasync def scrape(client, url):\\n    if not url:\\n        return None\\n    try:\\n        r = await client.get(url)\\n        return [\\n            line for line in r.text.splitlines() if line and not line.startswith(\\"#\\")\\n        ]\\n    except httpx.HTTPError as e:\\n        return f\\"scrape failed: {e}\\"\\n\\n\\nasync def run_cell(args, prompts, concurrency, repeat, env):\\n    n = args.requests or max(64, 4 * concurrency)\\n    n_warm = args.warmup if args.warmup is not None else max(2, concurrency)\\n    limits = httpx.Limits(\\n        max_connections=concurrency + 4, max_keepalive_connections=concurrency + 4\\n    )\\n    async with httpx.AsyncClient(\\n        timeout=httpx.Timeout(args.timeout), limits=limits\\n    ) as client:\\n        t0 = time.perf_counter()\\n        started = utcnow()\\n        # Warmup uses prompts from the end of the list so they do not prime the first measured prompts.\\n        warm = await closed_loop(\\n            client, args, prompts[::-1], n_warm, concurrency, True, t0\\n        )\\n        metrics_before = await scrape(client, args.metrics_url)\\n        gpu = GpuSampler()\\n        dcgm = DcgmSampler(args.dcgm_fields, args.dcgm_host)\\n        stats_task = asyncio.create_task(docker_stats())\\n        measured = await closed_loop(client, args, prompts, n, concurrency, False, t0)\\n        stats = await stats_task\\n        gpu_samples = gpu.stop()\\n        dcgm_samples, dcgm_error = dcgm.stop()\\n        metrics_after = await scrape(client, args.metrics_url)\\n    records = warm + measured\\n    return {\\n        \\"schema\\": 1,\\n        \\"smoke\\": args.smoke,\\n        \\"label\\": args.label,\\n        \\"engine\\": args.engine,\\n        \\"engine_version\\": args.engine_version,\\n        \\"model\\": args.model,\\n        \\"model_revision\\": args.model_revision,\\n        \\"precision\\": args.precision,\\n        \\"concurrency\\": concurrency,\\n        \\"repeat\\": repeat,\\n        \\"started_utc\\": started,\\n        \\"finished_utc\\": utcnow(),\\n        \\"config\\": {\\n            \\"base_url\\": args.base_url,\\n            \\"requests\\": n,\\n            \\"warmup_requests\\": n_warm,\\n            \\"max_tokens\\": args.max_tokens,\\n            \\"temperature\\": 0,\\n            \\"seed\\": args.seed,\\n            \\"ignore_eos\\": args.ignore_eos,\\n            \\"prompts_file\\": str(Path(args.prompts).name),\\n            \\"prompts_count\\": len(prompts),\\n            \\"prompts_sha256\\": args.prompts_sha256,\\n            \\"timeout_s\\": args.timeout,\\n            \\"notes\\": args.notes,\\n        },\\n        \\"env\\": env,\\n        \\"server_args\\": args.server_args,\\n        \\"gpu_samples\\": gpu_samples,\\n        \\"dcgm_samples\\": dcgm_samples,\\n        \\"dcgm_error\\": dcgm_error,\\n        \\"docker_stats\\": stats,\\n        \\"server_metrics_before\\": metrics_before,\\n        \\"server_metrics_after\\": metrics_after,\\n        \\"summary\\": summarize(records)\\n        | {\\"decode_checks\\": decode_checks(records, args.max_tokens, args.ignore_eos)},\\n        \\"requests\\": records,\\n    }\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(\\n        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter\\n    )\\n    p.add_argument(\\n        \\"--base-url\\",\\n        required=True,\\n        help=\\"OpenAI API root, e.g. http://127.0.0.1:8080/v1\\",\\n    )\\n    p.add_argument(\\"--model\\", required=True)\\n    p.add_argument(\\"--model-revision\\", default=None, help=\\"HF commit or file sha256\\")\\n    p.add_argument(\\"--engine\\", required=True)\\n    p.add_argument(\\"--engine-version\\", required=True)\\n    p.add_argument(\\"--precision\\", required=True)\\n    p.add_argument(\\"--label\\", required=True, help=\\"cell name prefix for output files\\")\\n    p.add_argument(\\n        \\"--hardware\\",\\n        default=None,\\n        help=\\"free-text hardware label, e.g. \'1x T4 (Kaggle)\'\\",\\n    )\\n    p.add_argument(\\"--concurrency\\", type=int, nargs=\\"+\\", default=[1])\\n    p.add_argument(\\"--repeats\\", type=int, default=3)\\n    p.add_argument(\\n        \\"--requests\\",\\n        type=int,\\n        default=0,\\n        help=\\"measured requests per run (0 = max(64, 4*concurrency))\\",\\n    )\\n    p.add_argument(\\n        \\"--warmup\\",\\n        type=int,\\n        default=None,\\n        help=\\"warmup requests per run (default max(2, concurrency))\\",\\n    )\\n    p.add_argument(\\"--max-tokens\\", type=int, default=128)\\n    p.add_argument(\\"--seed\\", type=int, default=1234)\\n    p.add_argument(\\"--no-ignore-eos\\", dest=\\"ignore_eos\\", action=\\"store_false\\")\\n    p.add_argument(\\"--timeout\\", type=float, default=600)\\n    p.add_argument(\\"--prompts\\", default=str(HERE / \\"prompts.jsonl\\"))\\n    p.add_argument(\\n        \\"--metrics-url\\",\\n        default=None,\\n        help=\\"Prometheus /metrics to snapshot before and after\\",\\n    )\\n    p.add_argument(\\n        \\"--server-args\\", default=None, help=\\"server command line, stored verbatim\\"\\n    )\\n    p.add_argument(\\n        \\"--git-commit\\", default=None, help=\\"override when not run from a git checkout\\"\\n    )\\n    p.add_argument(\\n        \\"--dcgm-fields\\",\\n        default=None,\\n        help=\\"comma-separated DCGM field ids to sample during the measured window (dcgm_sampler.py)\\",\\n    )\\n    p.add_argument(\\n        \\"--dcgm-host\\", default=\\"127.0.0.1\\", help=\\"nv-hostengine address, or \'embedded\'\\"\\n    )\\n    p.add_argument(\\"--notes\\", default=None)\\n    p.add_argument(\\n        \\"--smoke\\",\\n        action=\\"store_true\\",\\n        help=\\"mark as smoke run (excluded from RESULTS.md)\\",\\n    )\\n    p.add_argument(\\"--out-dir\\", required=True)\\n    args = p.parse_args()\\n\\n    raw_prompts = Path(args.prompts).read_bytes()\\n    args.prompts_sha256 = hashlib.sha256(raw_prompts).hexdigest()\\n    prompts = [\\n        json.loads(line)\\n        for line in raw_prompts.decode(\\"utf-8\\").splitlines()\\n        if line.strip()\\n    ]\\n    env = {\\n        \\"hardware\\": args.hardware,\\n        \\"cpu_model\\": cpu_model(),\\n        \\"cpu_count\\": os.cpu_count(),\\n        \\"platform\\": platform.platform(),\\n        \\"python\\": platform.python_version(),\\n        \\"httpx\\": httpx.__version__,\\n        \\"git_commit\\": args.git_commit or git_commit(),\\n        \\"nvidia_smi\\": sh(\\n            \\"nvidia-smi\\",\\n            \\"--query-gpu=name,driver_version,memory.total\\",\\n            \\"--format=csv,noheader\\",\\n        )\\n        if shutil.which(\\"nvidia-smi\\")\\n        else None,\\n    }\\n    with httpx.Client(timeout=30) as c:\\n        env[\\"server_models\\"] = c.get(f\\"{args.base_url}/models\\").json()\\n    out_dir = Path(args.out_dir)\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    for conc in args.concurrency:\\n        for rep in range(1, args.repeats + 1):\\n            res = asyncio.run(run_cell(args, prompts, conc, rep, env))\\n            stamp = (\\n                res[\\"started_utc\\"]\\n                .replace(\\":\\", \\"\\")\\n                .replace(\\"-\\", \\"\\")\\n                .replace(\\"+0000\\", \\"Z\\")\\n            )\\n            path = out_dir / f\\"{args.label}-c{conc}-r{rep}-{stamp}.json\\"\\n            path.write_text(\\n                json.dumps(res, separators=(\\",\\", \\":\\")) + \\"\\\\n\\",\\n                encoding=\\"utf-8\\",\\n                newline=\\"\\\\n\\",  # LF on Windows too\\n            )  # compact raw data\\n            s = res[\\"summary\\"]\\n            print(\\n                f\\"{path.name}: ok {s[\'ok\']}/{s[\'requests\']} out_tok/s {s[\'output_tok_s\']} \\"\\n                f\\"ttft_p50 {s[\'ttft_s\'][\'p50\']} e2e_p95 {s[\'e2e_s\'][\'p95\']}\\",\\n                flush=True,\\n            )\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"], "benchmark/loadgen/metrics.py": ["c58866fb6430059795c5cbeedc5a36ea0798c0c16b05d8161c160d43192fcc35", "\\"\\"\\"Pure metric functions. Times are seconds. Shared by run.py, report.py and the tests.\\"\\"\\"\\n\\nimport itertools\\nimport math\\nimport statistics\\n\\n\\ndef percentile(values, q):\\n    \\"\\"\\"Linear interpolation between closest ranks (same as numpy\'s default). None for empty input.\\"\\"\\"\\n    xs = sorted(values)\\n    if not xs:\\n        return None\\n    k = (len(xs) - 1) * q / 100\\n    lo, hi = math.floor(k), math.ceil(k)\\n    return xs[lo] + (xs[hi] - xs[lo]) * (k - lo)\\n\\n\\ndef dist(values):\\n    \\"\\"\\"mean, p50, p95, p99 of a list (all None if empty).\\"\\"\\"\\n    return {\\n        \\"mean\\": statistics.fmean(values) if values else None,\\n        \\"p50\\": percentile(values, 50),\\n        \\"p95\\": percentile(values, 95),\\n        \\"p99\\": percentile(values, 99),\\n    }\\n\\n\\ndef request_metrics(r):\\n    \\"\\"\\"TTFT, inter-token gaps, TPOT and e2e for one request record.\\n\\n    r has start, end, token_times (arrival time of each streamed content chunk) and output_tokens.\\n    ITL is the gap between consecutive chunks. TPOT is (last token - first token) / (output_tokens - 1),\\n    which stays correct when a chunk carries more than one token.\\n    \\"\\"\\"\\n    tt = r[\\"token_times\\"]\\n    ttft = tt[0] - r[\\"start\\"] if tt else None\\n    itl = [b - a for a, b in itertools.pairwise(tt)]\\n    n = r[\\"output_tokens\\"] or 0\\n    tpot = (tt[-1] - tt[0]) / (n - 1) if tt and n > 1 else None\\n    e2e = r[\\"end\\"] - r[\\"start\\"]\\n    return {\\n        \\"ttft\\": ttft,\\n        \\"itl\\": itl,\\n        \\"tpot\\": tpot,\\n        \\"e2e\\": e2e,\\n        \\"tok_s\\": n / e2e if e2e > 0 else None,\\n    }\\n\\n\\ndef summarize(records):\\n    \\"\\"\\"Summary over measured (non-warmup) requests.\\"\\"\\"\\n    measured = [r for r in records if not r[\\"warmup\\"]]\\n    ok = [r for r in measured if not r[\\"error\\"]]\\n    per = [request_metrics(r) for r in ok]\\n\\n    def pick(k):\\n        return [m[k] for m in per if m[k] is not None]\\n\\n    wall = (\\n        (max(r[\\"end\\"] for r in measured) - min(r[\\"start\\"] for r in measured))\\n        if measured\\n        else 0\\n    )\\n    out_tokens = sum(r[\\"output_tokens\\"] or 0 for r in ok)\\n    warm = [\\n        request_metrics(r)[\\"ttft\\"]\\n        for r in records\\n        if r[\\"warmup\\"] and not r[\\"error\\"] and r[\\"token_times\\"]\\n    ]\\n    return {\\n        \\"requests\\": len(measured),\\n        \\"ok\\": len(ok),\\n        \\"errors\\": len(measured) - len(ok),\\n        \\"error_rate\\": (len(measured) - len(ok)) / len(measured) if measured else None,\\n        \\"wall_s\\": wall,\\n        \\"output_tokens\\": out_tokens,\\n        \\"output_tok_s\\": out_tokens / wall if wall > 0 else None,\\n        \\"request_s\\": len(ok) / wall if wall > 0 else None,\\n        \\"ttft_s\\": dist(pick(\\"ttft\\")),\\n        \\"itl_s\\": dist([g for m in per for g in m[\\"itl\\"]]),\\n        \\"tpot_s\\": dist(pick(\\"tpot\\")),\\n        \\"e2e_s\\": dist(pick(\\"e2e\\")),\\n        \\"per_request_tok_s\\": dist(pick(\\"tok_s\\")),\\n        \\"warmup_ttft_mean_s\\": statistics.fmean(warm) if warm else None,\\n    }\\n\\n\\nTAIL_GAP_S = 0.25  # last streamed chunk this long before the response ended ...\\nTAIL_GAP_ITL = 10  # ... and this many times the request\'s median chunk gap = the stream went quiet early\\n\\n\\ndef stream_truncated(r):\\n    \\"\\"\\"True if the server stopped streaming text before the request ended: fewer chunks than output tokens and a\\n    long silent tail. Such a request\'s TPOT and ITL undercount (its tokens arrived in no streamed chunk).\\"\\"\\"\\n    tt, n = r[\\"token_times\\"], r.get(\\"output_tokens\\") or 0\\n    if not tt or len(tt) >= n:\\n        return False\\n    gaps = sorted(b - a for a, b in itertools.pairwise(tt))\\n    median = gaps[len(gaps) // 2] if gaps else 0\\n    tail = r[\\"end\\"] - tt[-1]\\n    return tail > TAIL_GAP_S and tail > TAIL_GAP_ITL * median\\n\\n\\ndef decode_checks(records, max_tokens, ignore_eos):\\n    \\"\\"\\"Over measured successful requests: how many decoded a token count other than max_tokens (only checked with\\n    ignore_eos, where every request must decode exactly max_tokens), and how many streams were truncated.\\"\\"\\"\\n    ok = [r for r in records if not r[\\"warmup\\"] and not r[\\"error\\"]]\\n    return {\\n        \\"checked\\": len(ok),\\n        \\"output_tokens_not_max\\": sum(\\n            1 for r in ok if r.get(\\"output_tokens\\") != max_tokens\\n        )\\n        if ignore_eos\\n        else None,\\n        \\"streams_truncated\\": sum(1 for r in ok if stream_truncated(r)),\\n    }\\n\\n\\ndef cv(values):\\n    \\"\\"\\"Coefficient of variation (stdev / mean) across repeats; None if under 2 values or mean 0.\\"\\"\\"\\n    xs = [v for v in values if v is not None]\\n    if len(xs) < 2 or statistics.fmean(xs) == 0:\\n        return None\\n    return statistics.stdev(xs) / abs(statistics.fmean(xs))\\n"], "benchmark/loadgen/prompts.jsonl": ["72fb01ab947799b7fcc757f90652501b203bff4bd445d3654d36aff01913ea7e", "{\\"id\\": \\"l06\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 6:\\\\nDay 1, shift 2: conveyor CV-3, vibration rose to 9.1 mm/s.\\\\nDay 2, shift 1: fan F-12, alignment checked, offset 0.55 mm.\\\\nDay 3, shift 3: pump P-101, alignment checked, offset 0.38 mm.\\\\nDay 4, shift 1: boiler B-2, vibration rose to 12.0 mm/s.\\\\nDay 5, shift 3: boiler B-2, bearing temperature reached 76 C.\\\\nDay 6, shift 3: mixer M-5, alignment checked, offset 0.4 mm.\\\\nDay 7, shift 2: compressor C-7, bearing temperature reached 90 C.\\\\nDay 8, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 9, shift 1: pump P-101, motor current 9 A above baseline.\\\\nDay 10, shift 2: pump P-101, bearing temperature reached 78 C.\\\\nDay 11, shift 2: conveyor CV-3, vibration rose to 2.6 mm/s.\\\\nDay 12, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 13, shift 2: compressor C-7, alignment checked, offset 0.57 mm.\\\\nDay 14, shift 3: mixer M-5, motor current 5 A above baseline.\\\\nDay 15, shift 2: compressor C-7, operator reported unusual noise.\\\\nDay 16, shift 3: mixer M-5, bearing temperature reached 91 C.\\\\nDay 17, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 18, shift 3: fan F-12, motor current 4 A above baseline.\\\\nDay 19, shift 3: boiler B-2, filter replaced during scheduled stop.\\\\nDay 20, shift 3: press PR-9, vibration rose to 2.7 mm/s.\\\\nDay 21, shift 3: fan F-12, motor current 15 A above baseline.\\\\nDay 22, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 23, shift 2: fan F-12, operator reported unusual noise.\\\\nDay 24, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 25, shift 3: compressor C-7, bearing temperature reached 64 C.\\\\nDay 26, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 27, shift 3: fan F-12, motor current 15 A above baseline.\\\\nDay 28, shift 1: mixer M-5, seal leak observed, 17 drops per minute.\\\\nDay 29, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 30, shift 3: chiller CH-4, vibration rose to 11.8 mm/s.\\\\nDay 31, shift 2: conveyor CV-3, operator reported unusual noise.\\\\nDay 32, shift 1: conveyor CV-3, oil level checked and topped up.\\\\nDay 33, shift 2: boiler B-2, vibration rose to 6.7 mm/s.\\\\nDay 34, shift 2: chiller CH-4, oil level checked and topped up.\\\\nDay 35, shift 2: chiller CH-4, seal leak observed, 3 drops per minute.\\\\nDay 36, shift 2: conveyor CV-3, alignment checked, offset 0.21 mm.\\\\n\\\\nWhat maintenance actions would you schedule next week?\\"}\\n{\\"id\\": \\"s09\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is vibration analysis used for?\\"}\\n{\\"id\\": \\"s33\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a spare parts criticality ranking?\\"}\\n{\\"id\\": \\"l13\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 13:\\\\nDay 1, shift 1: compressor C-7, oil level checked and topped up.\\\\nDay 2, shift 3: boiler B-2, alignment checked, offset 0.12 mm.\\\\nDay 3, shift 3: chiller CH-4, vibration rose to 11.6 mm/s.\\\\nDay 4, shift 1: compressor C-7, oil level checked and topped up.\\\\nDay 5, shift 1: compressor C-7, alignment checked, offset 0.08 mm.\\\\nDay 6, shift 3: pump P-101, alignment checked, offset 0.3 mm.\\\\nDay 7, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 8, shift 3: fan F-12, vibration rose to 8.9 mm/s.\\\\nDay 9, shift 3: chiller CH-4, bearing temperature reached 82 C.\\\\nDay 10, shift 3: pump P-101, seal leak observed, 2 drops per minute.\\\\nDay 11, shift 3: press PR-9, motor current 8 A above baseline.\\\\nDay 12, shift 1: boiler B-2, bearing temperature reached 60 C.\\\\nDay 13, shift 2: press PR-9, filter replaced during scheduled stop.\\\\nDay 14, shift 3: chiller CH-4, bearing temperature reached 79 C.\\\\nDay 15, shift 3: chiller CH-4, motor current 7 A above baseline.\\\\nDay 16, shift 1: boiler B-2, operator reported unusual noise.\\\\nDay 17, shift 1: mixer M-5, alignment checked, offset 0.28 mm.\\\\nDay 18, shift 3: press PR-9, bearing temperature reached 80 C.\\\\nDay 19, shift 1: press PR-9, vibration rose to 7.6 mm/s.\\\\nDay 20, shift 3: press PR-9, seal leak observed, 1 drops per minute.\\\\nDay 21, shift 3: compressor C-7, alignment checked, offset 0.46 mm.\\\\nDay 22, shift 2: pump P-101, bearing temperature reached 91 C.\\\\nDay 23, shift 3: boiler B-2, filter replaced during scheduled stop.\\\\nDay 24, shift 3: boiler B-2, motor current 2 A above baseline.\\\\nDay 25, shift 1: chiller CH-4, alignment checked, offset 0.1 mm.\\\\nDay 26, shift 3: boiler B-2, filter replaced during scheduled stop.\\\\nDay 27, shift 1: conveyor CV-3, motor current 6 A above baseline.\\\\nDay 28, shift 2: fan F-12, operator reported unusual noise.\\\\nDay 29, shift 3: boiler B-2, alignment checked, offset 0.1 mm.\\\\nDay 30, shift 3: boiler B-2, vibration rose to 11.3 mm/s.\\\\nDay 31, shift 3: boiler B-2, operator reported unusual noise.\\\\nDay 32, shift 1: conveyor CV-3, vibration rose to 6.7 mm/s.\\\\nDay 33, shift 1: compressor C-7, alignment checked, offset 0.5 mm.\\\\nDay 34, shift 2: compressor C-7, motor current 10 A above baseline.\\\\nDay 35, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 36, shift 1: compressor C-7, vibration rose to 6.2 mm/s.\\\\n\\\\nSummarise the main trends in this log.\\"}\\n{\\"id\\": \\"s06\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"List three causes of motor overheating.\\"}\\n{\\"id\\": \\"s41\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is oil analysis used for?\\"}\\n{\\"id\\": \\"s03\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Describe the difference between preventive and predictive maintenance.\\"}\\n{\\"id\\": \\"s08\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain lockout tagout in one paragraph.\\"}\\n{\\"id\\": \\"s35\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a thermocouple?\\"}\\n{\\"id\\": \\"l12\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 12:\\\\nDay 1, shift 2: boiler B-2, operator reported unusual noise.\\\\nDay 2, shift 2: compressor C-7, operator reported unusual noise.\\\\nDay 3, shift 2: boiler B-2, oil level checked and topped up.\\\\nDay 4, shift 2: conveyor CV-3, bearing temperature reached 62 C.\\\\nDay 5, shift 1: mixer M-5, operator reported unusual noise.\\\\nDay 6, shift 1: conveyor CV-3, bearing temperature reached 92 C.\\\\nDay 7, shift 1: compressor C-7, vibration rose to 6.7 mm/s.\\\\nDay 8, shift 3: pump P-101, motor current 10 A above baseline.\\\\nDay 9, shift 2: fan F-12, filter replaced during scheduled stop.\\\\nDay 10, shift 1: fan F-12, bearing temperature reached 87 C.\\\\nDay 11, shift 1: conveyor CV-3, operator reported unusual noise.\\\\nDay 12, shift 2: mixer M-5, operator reported unusual noise.\\\\nDay 13, shift 1: conveyor CV-3, vibration rose to 9.7 mm/s.\\\\nDay 14, shift 1: conveyor CV-3, oil level checked and topped up.\\\\nDay 15, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 16, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 17, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 18, shift 2: compressor C-7, seal leak observed, 7 drops per minute.\\\\nDay 19, shift 1: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 20, shift 2: press PR-9, seal leak observed, 3 drops per minute.\\\\nDay 21, shift 2: fan F-12, operator reported unusual noise.\\\\nDay 22, shift 1: conveyor CV-3, vibration rose to 11.0 mm/s.\\\\nDay 23, shift 3: chiller CH-4, operator reported unusual noise.\\\\nDay 24, shift 2: pump P-101, filter replaced during scheduled stop.\\\\nDay 25, shift 2: press PR-9, motor current 12 A above baseline.\\\\nDay 26, shift 3: chiller CH-4, alignment checked, offset 0.37 mm.\\\\nDay 27, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 28, shift 2: boiler B-2, bearing temperature reached 63 C.\\\\nDay 29, shift 3: boiler B-2, bearing temperature reached 92 C.\\\\nDay 30, shift 3: fan F-12, filter replaced during scheduled stop.\\\\nDay 31, shift 3: boiler B-2, oil level checked and topped up.\\\\nDay 32, shift 1: compressor C-7, operator reported unusual noise.\\\\nDay 33, shift 3: fan F-12, motor current 2 A above baseline.\\\\nDay 34, shift 3: press PR-9, oil level checked and topped up.\\\\nDay 35, shift 3: conveyor CV-3, alignment checked, offset 0.57 mm.\\\\nDay 36, shift 1: conveyor CV-3, motor current 6 A above baseline.\\\\n\\\\nWhich asset is most at risk of failure and why?\\"}\\n{\\"id\\": \\"l08\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 8:\\\\nDay 1, shift 2: pump P-101, motor current 4 A above baseline.\\\\nDay 2, shift 2: fan F-12, oil level checked and topped up.\\\\nDay 3, shift 1: chiller CH-4, motor current 5 A above baseline.\\\\nDay 4, shift 3: fan F-12, operator reported unusual noise.\\\\nDay 5, shift 3: conveyor CV-3, oil level checked and topped up.\\\\nDay 6, shift 3: press PR-9, vibration rose to 3.1 mm/s.\\\\nDay 7, shift 2: conveyor CV-3, operator reported unusual noise.\\\\nDay 8, shift 2: boiler B-2, bearing temperature reached 74 C.\\\\nDay 9, shift 2: conveyor CV-3, vibration rose to 5.8 mm/s.\\\\nDay 10, shift 1: mixer M-5, motor current 12 A above baseline.\\\\nDay 11, shift 3: boiler B-2, motor current 13 A above baseline.\\\\nDay 12, shift 2: fan F-12, oil level checked and topped up.\\\\nDay 13, shift 1: fan F-12, oil level checked and topped up.\\\\nDay 14, shift 1: mixer M-5, oil level checked and topped up.\\\\nDay 15, shift 1: fan F-12, bearing temperature reached 81 C.\\\\nDay 16, shift 3: chiller CH-4, vibration rose to 7.4 mm/s.\\\\nDay 17, shift 3: boiler B-2, vibration rose to 11.1 mm/s.\\\\nDay 18, shift 1: conveyor CV-3, vibration rose to 9.7 mm/s.\\\\nDay 19, shift 2: mixer M-5, motor current 13 A above baseline.\\\\nDay 20, shift 3: fan F-12, alignment checked, offset 0.37 mm.\\\\nDay 21, shift 3: compressor C-7, bearing temperature reached 89 C.\\\\nDay 22, shift 3: mixer M-5, motor current 10 A above baseline.\\\\nDay 23, shift 3: mixer M-5, alignment checked, offset 0.56 mm.\\\\nDay 24, shift 2: compressor C-7, bearing temperature reached 73 C.\\\\nDay 25, shift 1: conveyor CV-3, alignment checked, offset 0.19 mm.\\\\nDay 26, shift 2: press PR-9, bearing temperature reached 93 C.\\\\nDay 27, shift 1: chiller CH-4, bearing temperature reached 89 C.\\\\nDay 28, shift 1: press PR-9, seal leak observed, 4 drops per minute.\\\\nDay 29, shift 2: conveyor CV-3, bearing temperature reached 79 C.\\\\nDay 30, shift 2: chiller CH-4, motor current 6 A above baseline.\\\\nDay 31, shift 3: mixer M-5, operator reported unusual noise.\\\\nDay 32, shift 3: press PR-9, seal leak observed, 7 drops per minute.\\\\nDay 33, shift 3: fan F-12, vibration rose to 2.2 mm/s.\\\\nDay 34, shift 2: chiller CH-4, motor current 3 A above baseline.\\\\nDay 35, shift 2: conveyor CV-3, operator reported unusual noise.\\\\nDay 36, shift 1: chiller CH-4, bearing temperature reached 69 C.\\\\n\\\\nWhich asset is most at risk of failure and why?\\"}\\n{\\"id\\": \\"s00\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the purpose of a pressure relief valve?\\"}\\n{\\"id\\": \\"s40\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Describe what a borescope inspection is.\\"}\\n{\\"id\\": \\"l01\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 1:\\\\nDay 1, shift 3: pump P-101, motor current 4 A above baseline.\\\\nDay 2, shift 3: conveyor CV-3, bearing temperature reached 70 C.\\\\nDay 3, shift 2: mixer M-5, motor current 11 A above baseline.\\\\nDay 4, shift 1: pump P-101, motor current 14 A above baseline.\\\\nDay 5, shift 2: compressor C-7, motor current 7 A above baseline.\\\\nDay 6, shift 2: press PR-9, vibration rose to 2.3 mm/s.\\\\nDay 7, shift 3: conveyor CV-3, alignment checked, offset 0.07 mm.\\\\nDay 8, shift 1: fan F-12, motor current 12 A above baseline.\\\\nDay 9, shift 3: conveyor CV-3, operator reported unusual noise.\\\\nDay 10, shift 2: press PR-9, motor current 6 A above baseline.\\\\nDay 11, shift 3: boiler B-2, filter replaced during scheduled stop.\\\\nDay 12, shift 2: pump P-101, oil level checked and topped up.\\\\nDay 13, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 14, shift 2: mixer M-5, oil level checked and topped up.\\\\nDay 15, shift 2: mixer M-5, alignment checked, offset 0.31 mm.\\\\nDay 16, shift 1: fan F-12, bearing temperature reached 88 C.\\\\nDay 17, shift 2: compressor C-7, alignment checked, offset 0.49 mm.\\\\nDay 18, shift 3: boiler B-2, bearing temperature reached 73 C.\\\\nDay 19, shift 1: boiler B-2, alignment checked, offset 0.56 mm.\\\\nDay 20, shift 2: press PR-9, bearing temperature reached 66 C.\\\\nDay 21, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 22, shift 3: pump P-101, filter replaced during scheduled stop.\\\\nDay 23, shift 2: conveyor CV-3, seal leak observed, 20 drops per minute.\\\\nDay 24, shift 2: fan F-12, oil level checked and topped up.\\\\nDay 25, shift 2: fan F-12, filter replaced during scheduled stop.\\\\nDay 26, shift 1: fan F-12, oil level checked and topped up.\\\\nDay 27, shift 3: boiler B-2, motor current 8 A above baseline.\\\\nDay 28, shift 3: mixer M-5, bearing temperature reached 59 C.\\\\nDay 29, shift 2: mixer M-5, alignment checked, offset 0.51 mm.\\\\nDay 30, shift 1: press PR-9, bearing temperature reached 56 C.\\\\nDay 31, shift 2: conveyor CV-3, seal leak observed, 30 drops per minute.\\\\nDay 32, shift 2: fan F-12, operator reported unusual noise.\\\\nDay 33, shift 2: conveyor CV-3, bearing temperature reached 69 C.\\\\nDay 34, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 35, shift 2: boiler B-2, motor current 8 A above baseline.\\\\nDay 36, shift 2: boiler B-2, oil level checked and topped up.\\\\n\\\\nSummarise the main trends in this log.\\"}\\n{\\"id\\": \\"l02\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 2:\\\\nDay 1, shift 3: boiler B-2, vibration rose to 6.1 mm/s.\\\\nDay 2, shift 3: boiler B-2, alignment checked, offset 0.21 mm.\\\\nDay 3, shift 2: compressor C-7, alignment checked, offset 0.17 mm.\\\\nDay 4, shift 3: mixer M-5, bearing temperature reached 66 C.\\\\nDay 5, shift 1: fan F-12, operator reported unusual noise.\\\\nDay 6, shift 2: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 7, shift 1: pump P-101, vibration rose to 6.1 mm/s.\\\\nDay 8, shift 2: boiler B-2, alignment checked, offset 0.3 mm.\\\\nDay 9, shift 2: conveyor CV-3, seal leak observed, 29 drops per minute.\\\\nDay 10, shift 3: fan F-12, oil level checked and topped up.\\\\nDay 11, shift 1: press PR-9, vibration rose to 6.2 mm/s.\\\\nDay 12, shift 3: compressor C-7, seal leak observed, 23 drops per minute.\\\\nDay 13, shift 2: mixer M-5, alignment checked, offset 0.55 mm.\\\\nDay 14, shift 1: boiler B-2, bearing temperature reached 75 C.\\\\nDay 15, shift 3: fan F-12, alignment checked, offset 0.38 mm.\\\\nDay 16, shift 2: pump P-101, motor current 5 A above baseline.\\\\nDay 17, shift 3: conveyor CV-3, vibration rose to 4.7 mm/s.\\\\nDay 18, shift 1: pump P-101, motor current 13 A above baseline.\\\\nDay 19, shift 3: compressor C-7, filter replaced during scheduled stop.\\\\nDay 20, shift 1: fan F-12, seal leak observed, 15 drops per minute.\\\\nDay 21, shift 1: fan F-12, motor current 13 A above baseline.\\\\nDay 22, shift 1: press PR-9, motor current 3 A above baseline.\\\\nDay 23, shift 2: pump P-101, motor current 11 A above baseline.\\\\nDay 24, shift 3: fan F-12, vibration rose to 6.2 mm/s.\\\\nDay 25, shift 2: boiler B-2, seal leak observed, 16 drops per minute.\\\\nDay 26, shift 3: press PR-9, seal leak observed, 5 drops per minute.\\\\nDay 27, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 28, shift 3: press PR-9, vibration rose to 4.6 mm/s.\\\\nDay 29, shift 2: pump P-101, vibration rose to 4.0 mm/s.\\\\nDay 30, shift 2: chiller CH-4, motor current 8 A above baseline.\\\\nDay 31, shift 2: pump P-101, motor current 3 A above baseline.\\\\nDay 32, shift 3: mixer M-5, filter replaced during scheduled stop.\\\\nDay 33, shift 1: conveyor CV-3, alignment checked, offset 0.57 mm.\\\\nDay 34, shift 1: pump P-101, seal leak observed, 27 drops per minute.\\\\nDay 35, shift 2: boiler B-2, seal leak observed, 30 drops per minute.\\\\nDay 36, shift 1: boiler B-2, motor current 12 A above baseline.\\\\n\\\\nWhat maintenance actions would you schedule next week?\\"}\\n{\\"id\\": \\"l11\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 11:\\\\nDay 1, shift 1: fan F-12, motor current 8 A above baseline.\\\\nDay 2, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 3, shift 3: pump P-101, motor current 3 A above baseline.\\\\nDay 4, shift 2: compressor C-7, vibration rose to 6.1 mm/s.\\\\nDay 5, shift 1: compressor C-7, bearing temperature reached 84 C.\\\\nDay 6, shift 3: conveyor CV-3, oil level checked and topped up.\\\\nDay 7, shift 3: mixer M-5, seal leak observed, 12 drops per minute.\\\\nDay 8, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 9, shift 3: pump P-101, operator reported unusual noise.\\\\nDay 10, shift 2: mixer M-5, motor current 7 A above baseline.\\\\nDay 11, shift 2: conveyor CV-3, motor current 11 A above baseline.\\\\nDay 12, shift 1: compressor C-7, motor current 6 A above baseline.\\\\nDay 13, shift 2: mixer M-5, bearing temperature reached 86 C.\\\\nDay 14, shift 1: press PR-9, alignment checked, offset 0.31 mm.\\\\nDay 15, shift 2: pump P-101, motor current 4 A above baseline.\\\\nDay 16, shift 3: conveyor CV-3, operator reported unusual noise.\\\\nDay 17, shift 2: boiler B-2, vibration rose to 5.9 mm/s.\\\\nDay 18, shift 3: compressor C-7, oil level checked and topped up.\\\\nDay 19, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 20, shift 3: pump P-101, bearing temperature reached 70 C.\\\\nDay 21, shift 3: conveyor CV-3, oil level checked and topped up.\\\\nDay 22, shift 2: chiller CH-4, alignment checked, offset 0.09 mm.\\\\nDay 23, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 24, shift 3: chiller CH-4, motor current 7 A above baseline.\\\\nDay 25, shift 2: press PR-9, vibration rose to 9.1 mm/s.\\\\nDay 26, shift 3: compressor C-7, motor current 15 A above baseline.\\\\nDay 27, shift 1: fan F-12, alignment checked, offset 0.5 mm.\\\\nDay 28, shift 1: conveyor CV-3, operator reported unusual noise.\\\\nDay 29, shift 1: mixer M-5, bearing temperature reached 92 C.\\\\nDay 30, shift 1: fan F-12, seal leak observed, 28 drops per minute.\\\\nDay 31, shift 2: mixer M-5, bearing temperature reached 87 C.\\\\nDay 32, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 33, shift 2: fan F-12, bearing temperature reached 56 C.\\\\nDay 34, shift 2: chiller CH-4, motor current 11 A above baseline.\\\\nDay 35, shift 1: press PR-9, bearing temperature reached 66 C.\\\\nDay 36, shift 3: press PR-9, alignment checked, offset 0.11 mm.\\\\n\\\\nList every asset with a temperature or vibration problem.\\"}\\n{\\"id\\": \\"s44\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain what a Gantt chart shows in a shutdown plan.\\"}\\n{\\"id\\": \\"s29\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the difference between accuracy and precision of a sensor?\\"}\\n{\\"id\\": \\"s30\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Why do pipes experience water hammer?\\"}\\n{\\"id\\": \\"s07\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the function of a heat exchanger?\\"}\\n{\\"id\\": \\"s38\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain what shaft misalignment causes.\\"}\\n{\\"id\\": \\"s19\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a failure mode and effects analysis?\\"}\\n{\\"id\\": \\"l03\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 3:\\\\nDay 1, shift 1: chiller CH-4, operator reported unusual noise.\\\\nDay 2, shift 1: press PR-9, alignment checked, offset 0.22 mm.\\\\nDay 3, shift 2: mixer M-5, operator reported unusual noise.\\\\nDay 4, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 5, shift 1: press PR-9, seal leak observed, 28 drops per minute.\\\\nDay 6, shift 1: conveyor CV-3, operator reported unusual noise.\\\\nDay 7, shift 2: compressor C-7, oil level checked and topped up.\\\\nDay 8, shift 2: chiller CH-4, motor current 7 A above baseline.\\\\nDay 9, shift 3: mixer M-5, motor current 9 A above baseline.\\\\nDay 10, shift 3: conveyor CV-3, operator reported unusual noise.\\\\nDay 11, shift 2: fan F-12, motor current 8 A above baseline.\\\\nDay 12, shift 1: mixer M-5, bearing temperature reached 63 C.\\\\nDay 13, shift 3: mixer M-5, vibration rose to 6.1 mm/s.\\\\nDay 14, shift 3: compressor C-7, bearing temperature reached 78 C.\\\\nDay 15, shift 3: pump P-101, vibration rose to 8.8 mm/s.\\\\nDay 16, shift 2: compressor C-7, seal leak observed, 8 drops per minute.\\\\nDay 17, shift 2: compressor C-7, bearing temperature reached 73 C.\\\\nDay 18, shift 1: press PR-9, vibration rose to 6.6 mm/s.\\\\nDay 19, shift 1: boiler B-2, filter replaced during scheduled stop.\\\\nDay 20, shift 2: chiller CH-4, seal leak observed, 2 drops per minute.\\\\nDay 21, shift 1: mixer M-5, operator reported unusual noise.\\\\nDay 22, shift 3: compressor C-7, filter replaced during scheduled stop.\\\\nDay 23, shift 1: pump P-101, oil level checked and topped up.\\\\nDay 24, shift 1: compressor C-7, bearing temperature reached 57 C.\\\\nDay 25, shift 1: fan F-12, motor current 10 A above baseline.\\\\nDay 26, shift 3: press PR-9, alignment checked, offset 0.4 mm.\\\\nDay 27, shift 2: fan F-12, filter replaced during scheduled stop.\\\\nDay 28, shift 2: fan F-12, bearing temperature reached 61 C.\\\\nDay 29, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 30, shift 1: fan F-12, alignment checked, offset 0.19 mm.\\\\nDay 31, shift 1: compressor C-7, bearing temperature reached 80 C.\\\\nDay 32, shift 2: chiller CH-4, oil level checked and topped up.\\\\nDay 33, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 34, shift 2: mixer M-5, vibration rose to 4.0 mm/s.\\\\nDay 35, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 36, shift 2: fan F-12, alignment checked, offset 0.42 mm.\\\\n\\\\nList every asset with a temperature or vibration problem.\\"}\\n{\\"id\\": \\"s22\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"How is corrosion prevented in steel pipes?\\"}\\n{\\"id\\": \\"s14\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain the difference between AC and DC motors.\\"}\\n{\\"id\\": \\"s21\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain what a safety integrity level is.\\"}\\n{\\"id\\": \\"s12\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Define overall equipment effectiveness.\\"}\\n{\\"id\\": \\"l14\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 14:\\\\nDay 1, shift 1: fan F-12, alignment checked, offset 0.22 mm.\\\\nDay 2, shift 1: boiler B-2, vibration rose to 8.7 mm/s.\\\\nDay 3, shift 3: pump P-101, operator reported unusual noise.\\\\nDay 4, shift 3: conveyor CV-3, oil level checked and topped up.\\\\nDay 5, shift 2: boiler B-2, vibration rose to 9.5 mm/s.\\\\nDay 6, shift 3: press PR-9, bearing temperature reached 81 C.\\\\nDay 7, shift 2: chiller CH-4, vibration rose to 9.3 mm/s.\\\\nDay 8, shift 3: pump P-101, filter replaced during scheduled stop.\\\\nDay 9, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 10, shift 2: fan F-12, seal leak observed, 8 drops per minute.\\\\nDay 11, shift 1: fan F-12, oil level checked and topped up.\\\\nDay 12, shift 3: chiller CH-4, bearing temperature reached 83 C.\\\\nDay 13, shift 2: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 14, shift 3: mixer M-5, motor current 15 A above baseline.\\\\nDay 15, shift 2: fan F-12, vibration rose to 11.6 mm/s.\\\\nDay 16, shift 2: press PR-9, bearing temperature reached 81 C.\\\\nDay 17, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 18, shift 1: compressor C-7, filter replaced during scheduled stop.\\\\nDay 19, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 20, shift 3: boiler B-2, bearing temperature reached 70 C.\\\\nDay 21, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 22, shift 1: press PR-9, seal leak observed, 6 drops per minute.\\\\nDay 23, shift 1: conveyor CV-3, seal leak observed, 21 drops per minute.\\\\nDay 24, shift 2: pump P-101, motor current 5 A above baseline.\\\\nDay 25, shift 2: fan F-12, operator reported unusual noise.\\\\nDay 26, shift 3: chiller CH-4, operator reported unusual noise.\\\\nDay 27, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 28, shift 1: conveyor CV-3, oil level checked and topped up.\\\\nDay 29, shift 2: compressor C-7, alignment checked, offset 0.47 mm.\\\\nDay 30, shift 1: press PR-9, filter replaced during scheduled stop.\\\\nDay 31, shift 3: chiller CH-4, motor current 4 A above baseline.\\\\nDay 32, shift 1: conveyor CV-3, bearing temperature reached 92 C.\\\\nDay 33, shift 3: compressor C-7, oil level checked and topped up.\\\\nDay 34, shift 1: compressor C-7, filter replaced during scheduled stop.\\\\nDay 35, shift 2: press PR-9, bearing temperature reached 61 C.\\\\nDay 36, shift 3: fan F-12, operator reported unusual noise.\\\\n\\\\nWhat maintenance actions would you schedule next week?\\"}\\n{\\"id\\": \\"s15\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a root cause analysis?\\"}\\n{\\"id\\": \\"s32\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Describe what a calibration certificate contains.\\"}\\n{\\"id\\": \\"s36\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"How does an ultrasonic flow meter work?\\"}\\n{\\"id\\": \\"s45\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Why is grounding important in electrical panels?\\"}\\n{\\"id\\": \\"s16\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Why is lubrication important for gearboxes?\\"}\\n{\\"id\\": \\"l00\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 0:\\\\nDay 1, shift 1: press PR-9, bearing temperature reached 92 C.\\\\nDay 2, shift 3: fan F-12, oil level checked and topped up.\\\\nDay 3, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 4, shift 3: compressor C-7, alignment checked, offset 0.08 mm.\\\\nDay 5, shift 1: compressor C-7, filter replaced during scheduled stop.\\\\nDay 6, shift 2: press PR-9, vibration rose to 11.4 mm/s.\\\\nDay 7, shift 1: mixer M-5, operator reported unusual noise.\\\\nDay 8, shift 2: conveyor CV-3, vibration rose to 11.4 mm/s.\\\\nDay 9, shift 1: pump P-101, seal leak observed, 9 drops per minute.\\\\nDay 10, shift 2: press PR-9, vibration rose to 7.6 mm/s.\\\\nDay 11, shift 1: press PR-9, vibration rose to 9.8 mm/s.\\\\nDay 12, shift 1: press PR-9, alignment checked, offset 0.28 mm.\\\\nDay 13, shift 3: compressor C-7, alignment checked, offset 0.24 mm.\\\\nDay 14, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 15, shift 1: compressor C-7, seal leak observed, 2 drops per minute.\\\\nDay 16, shift 3: fan F-12, filter replaced during scheduled stop.\\\\nDay 17, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 18, shift 1: boiler B-2, motor current 10 A above baseline.\\\\nDay 19, shift 3: boiler B-2, filter replaced during scheduled stop.\\\\nDay 20, shift 3: press PR-9, motor current 11 A above baseline.\\\\nDay 21, shift 1: compressor C-7, motor current 12 A above baseline.\\\\nDay 22, shift 2: boiler B-2, vibration rose to 6.3 mm/s.\\\\nDay 23, shift 1: chiller CH-4, seal leak observed, 5 drops per minute.\\\\nDay 24, shift 2: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 25, shift 2: chiller CH-4, oil level checked and topped up.\\\\nDay 26, shift 2: boiler B-2, alignment checked, offset 0.31 mm.\\\\nDay 27, shift 2: mixer M-5, alignment checked, offset 0.2 mm.\\\\nDay 28, shift 3: fan F-12, oil level checked and topped up.\\\\nDay 29, shift 2: pump P-101, alignment checked, offset 0.58 mm.\\\\nDay 30, shift 2: compressor C-7, operator reported unusual noise.\\\\nDay 31, shift 3: chiller CH-4, motor current 11 A above baseline.\\\\nDay 32, shift 2: chiller CH-4, bearing temperature reached 93 C.\\\\nDay 33, shift 1: press PR-9, filter replaced during scheduled stop.\\\\nDay 34, shift 1: chiller CH-4, motor current 8 A above baseline.\\\\nDay 35, shift 1: boiler B-2, operator reported unusual noise.\\\\nDay 36, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\n\\\\nWhich asset is most at risk of failure and why?\\"}\\n{\\"id\\": \\"s25\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a condition monitoring sensor?\\"}\\n{\\"id\\": \\"s47\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain the purpose of a mean time to repair metric.\\"}\\n{\\"id\\": \\"s34\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain the bathtub curve of failure rates.\\"}\\n{\\"id\\": \\"s04\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a PID controller?\\"}\\n{\\"id\\": \\"s37\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a confined space permit?\\"}\\n{\\"id\\": \\"s27\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a Pareto chart used for in quality control?\\"}\\n{\\"id\\": \\"s39\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the purpose of an emergency stop circuit?\\"}\\n{\\"id\\": \\"s20\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What does a torque wrench do?\\"}\\n{\\"id\\": \\"l10\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 10:\\\\nDay 1, shift 2: fan F-12, oil level checked and topped up.\\\\nDay 2, shift 3: press PR-9, motor current 11 A above baseline.\\\\nDay 3, shift 1: pump P-101, oil level checked and topped up.\\\\nDay 4, shift 2: conveyor CV-3, motor current 6 A above baseline.\\\\nDay 5, shift 3: press PR-9, motor current 12 A above baseline.\\\\nDay 6, shift 2: mixer M-5, seal leak observed, 25 drops per minute.\\\\nDay 7, shift 3: fan F-12, bearing temperature reached 86 C.\\\\nDay 8, shift 1: chiller CH-4, seal leak observed, 19 drops per minute.\\\\nDay 9, shift 1: conveyor CV-3, operator reported unusual noise.\\\\nDay 10, shift 1: mixer M-5, alignment checked, offset 0.35 mm.\\\\nDay 11, shift 2: compressor C-7, motor current 4 A above baseline.\\\\nDay 12, shift 1: pump P-101, oil level checked and topped up.\\\\nDay 13, shift 3: compressor C-7, vibration rose to 7.8 mm/s.\\\\nDay 14, shift 3: chiller CH-4, bearing temperature reached 57 C.\\\\nDay 15, shift 2: mixer M-5, filter replaced during scheduled stop.\\\\nDay 16, shift 3: mixer M-5, vibration rose to 4.8 mm/s.\\\\nDay 17, shift 3: fan F-12, oil level checked and topped up.\\\\nDay 18, shift 1: boiler B-2, operator reported unusual noise.\\\\nDay 19, shift 3: pump P-101, operator reported unusual noise.\\\\nDay 20, shift 3: fan F-12, seal leak observed, 8 drops per minute.\\\\nDay 21, shift 2: fan F-12, vibration rose to 6.6 mm/s.\\\\nDay 22, shift 2: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 23, shift 2: compressor C-7, operator reported unusual noise.\\\\nDay 24, shift 2: mixer M-5, oil level checked and topped up.\\\\nDay 25, shift 3: conveyor CV-3, vibration rose to 2.7 mm/s.\\\\nDay 26, shift 1: mixer M-5, filter replaced during scheduled stop.\\\\nDay 27, shift 1: conveyor CV-3, alignment checked, offset 0.19 mm.\\\\nDay 28, shift 3: pump P-101, oil level checked and topped up.\\\\nDay 29, shift 1: fan F-12, motor current 9 A above baseline.\\\\nDay 30, shift 3: fan F-12, motor current 6 A above baseline.\\\\nDay 31, shift 3: chiller CH-4, alignment checked, offset 0.44 mm.\\\\nDay 32, shift 2: chiller CH-4, vibration rose to 4.6 mm/s.\\\\nDay 33, shift 1: pump P-101, seal leak observed, 7 drops per minute.\\\\nDay 34, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 35, shift 2: boiler B-2, operator reported unusual noise.\\\\nDay 36, shift 1: pump P-101, oil level checked and topped up.\\\\n\\\\nWhat maintenance actions would you schedule next week?\\"}\\n{\\"id\\": \\"s13\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is thermal imaging used for in maintenance?\\"}\\n{\\"id\\": \\"s17\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is SCADA?\\"}\\n{\\"id\\": \\"l15\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 15:\\\\nDay 1, shift 2: compressor C-7, oil level checked and topped up.\\\\nDay 2, shift 2: conveyor CV-3, alignment checked, offset 0.5 mm.\\\\nDay 3, shift 1: mixer M-5, seal leak observed, 4 drops per minute.\\\\nDay 4, shift 3: chiller CH-4, seal leak observed, 30 drops per minute.\\\\nDay 5, shift 2: press PR-9, seal leak observed, 9 drops per minute.\\\\nDay 6, shift 1: pump P-101, alignment checked, offset 0.55 mm.\\\\nDay 7, shift 3: pump P-101, oil level checked and topped up.\\\\nDay 8, shift 1: pump P-101, oil level checked and topped up.\\\\nDay 9, shift 3: compressor C-7, operator reported unusual noise.\\\\nDay 10, shift 3: compressor C-7, bearing temperature reached 75 C.\\\\nDay 11, shift 1: press PR-9, vibration rose to 7.1 mm/s.\\\\nDay 12, shift 2: mixer M-5, bearing temperature reached 92 C.\\\\nDay 13, shift 3: mixer M-5, seal leak observed, 14 drops per minute.\\\\nDay 14, shift 3: boiler B-2, operator reported unusual noise.\\\\nDay 15, shift 3: compressor C-7, filter replaced during scheduled stop.\\\\nDay 16, shift 2: chiller CH-4, seal leak observed, 23 drops per minute.\\\\nDay 17, shift 1: mixer M-5, oil level checked and topped up.\\\\nDay 18, shift 1: press PR-9, alignment checked, offset 0.5 mm.\\\\nDay 19, shift 1: boiler B-2, seal leak observed, 15 drops per minute.\\\\nDay 20, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 21, shift 3: boiler B-2, bearing temperature reached 69 C.\\\\nDay 22, shift 2: fan F-12, vibration rose to 9.6 mm/s.\\\\nDay 23, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 24, shift 3: pump P-101, seal leak observed, 23 drops per minute.\\\\nDay 25, shift 1: press PR-9, seal leak observed, 7 drops per minute.\\\\nDay 26, shift 1: press PR-9, vibration rose to 11.7 mm/s.\\\\nDay 27, shift 3: pump P-101, operator reported unusual noise.\\\\nDay 28, shift 1: chiller CH-4, seal leak observed, 15 drops per minute.\\\\nDay 29, shift 2: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 30, shift 3: pump P-101, operator reported unusual noise.\\\\nDay 31, shift 2: fan F-12, alignment checked, offset 0.5 mm.\\\\nDay 32, shift 2: chiller CH-4, oil level checked and topped up.\\\\nDay 33, shift 1: compressor C-7, filter replaced during scheduled stop.\\\\nDay 34, shift 1: fan F-12, bearing temperature reached 56 C.\\\\nDay 35, shift 1: mixer M-5, motor current 2 A above baseline.\\\\nDay 36, shift 3: press PR-9, alignment checked, offset 0.16 mm.\\\\n\\\\nList every asset with a temperature or vibration problem.\\"}\\n{\\"id\\": \\"s02\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What does MTBF stand for and how is it used?\\"}\\n{\\"id\\": \\"s05\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Why do bearings fail early?\\"}\\n{\\"id\\": \\"s28\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"How do you size a fuse for a motor circuit?\\"}\\n{\\"id\\": \\"s01\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain cavitation in centrifugal pumps.\\"}\\n{\\"id\\": \\"s31\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a work order in a CMMS?\\"}\\n{\\"id\\": \\"s26\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain what an interlock is.\\"}\\n{\\"id\\": \\"s43\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a hazard and operability study?\\"}\\n{\\"id\\": \\"s18\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Describe how a compressor works.\\"}\\n{\\"id\\": \\"s42\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"How do you read a P&ID diagram?\\"}\\n{\\"id\\": \\"l07\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 7:\\\\nDay 1, shift 1: conveyor CV-3, oil level checked and topped up.\\\\nDay 2, shift 3: press PR-9, seal leak observed, 13 drops per minute.\\\\nDay 3, shift 1: press PR-9, motor current 5 A above baseline.\\\\nDay 4, shift 3: conveyor CV-3, motor current 13 A above baseline.\\\\nDay 5, shift 1: conveyor CV-3, seal leak observed, 14 drops per minute.\\\\nDay 6, shift 1: fan F-12, vibration rose to 6.4 mm/s.\\\\nDay 7, shift 3: compressor C-7, operator reported unusual noise.\\\\nDay 8, shift 2: mixer M-5, operator reported unusual noise.\\\\nDay 9, shift 1: fan F-12, operator reported unusual noise.\\\\nDay 10, shift 2: conveyor CV-3, motor current 4 A above baseline.\\\\nDay 11, shift 2: compressor C-7, operator reported unusual noise.\\\\nDay 12, shift 1: pump P-101, alignment checked, offset 0.5 mm.\\\\nDay 13, shift 2: chiller CH-4, seal leak observed, 19 drops per minute.\\\\nDay 14, shift 3: conveyor CV-3, alignment checked, offset 0.11 mm.\\\\nDay 15, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 16, shift 3: boiler B-2, vibration rose to 3.3 mm/s.\\\\nDay 17, shift 2: boiler B-2, seal leak observed, 5 drops per minute.\\\\nDay 18, shift 3: mixer M-5, seal leak observed, 24 drops per minute.\\\\nDay 19, shift 1: conveyor CV-3, operator reported unusual noise.\\\\nDay 20, shift 1: pump P-101, vibration rose to 10.0 mm/s.\\\\nDay 21, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 22, shift 1: compressor C-7, oil level checked and topped up.\\\\nDay 23, shift 3: pump P-101, motor current 11 A above baseline.\\\\nDay 24, shift 1: conveyor CV-3, oil level checked and topped up.\\\\nDay 25, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 26, shift 2: conveyor CV-3, alignment checked, offset 0.33 mm.\\\\nDay 27, shift 2: chiller CH-4, bearing temperature reached 87 C.\\\\nDay 28, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 29, shift 2: compressor C-7, seal leak observed, 10 drops per minute.\\\\nDay 30, shift 3: chiller CH-4, vibration rose to 8.4 mm/s.\\\\nDay 31, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 32, shift 1: pump P-101, alignment checked, offset 0.08 mm.\\\\nDay 33, shift 1: press PR-9, seal leak observed, 2 drops per minute.\\\\nDay 34, shift 1: conveyor CV-3, alignment checked, offset 0.19 mm.\\\\nDay 35, shift 3: mixer M-5, motor current 12 A above baseline.\\\\nDay 36, shift 2: press PR-9, seal leak observed, 3 drops per minute.\\\\n\\\\nList every asset with a temperature or vibration problem.\\"}\\n{\\"id\\": \\"l05\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 5:\\\\nDay 1, shift 2: pump P-101, bearing temperature reached 94 C.\\\\nDay 2, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 3, shift 1: mixer M-5, filter replaced during scheduled stop.\\\\nDay 4, shift 3: mixer M-5, operator reported unusual noise.\\\\nDay 5, shift 3: press PR-9, bearing temperature reached 65 C.\\\\nDay 6, shift 1: conveyor CV-3, vibration rose to 9.9 mm/s.\\\\nDay 7, shift 3: fan F-12, motor current 5 A above baseline.\\\\nDay 8, shift 2: conveyor CV-3, seal leak observed, 16 drops per minute.\\\\nDay 9, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 10, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 11, shift 3: pump P-101, oil level checked and topped up.\\\\nDay 12, shift 3: fan F-12, motor current 14 A above baseline.\\\\nDay 13, shift 1: chiller CH-4, vibration rose to 7.7 mm/s.\\\\nDay 14, shift 3: compressor C-7, vibration rose to 2.5 mm/s.\\\\nDay 15, shift 2: press PR-9, motor current 15 A above baseline.\\\\nDay 16, shift 1: chiller CH-4, motor current 13 A above baseline.\\\\nDay 17, shift 2: press PR-9, filter replaced during scheduled stop.\\\\nDay 18, shift 2: pump P-101, filter replaced during scheduled stop.\\\\nDay 19, shift 3: chiller CH-4, vibration rose to 7.8 mm/s.\\\\nDay 20, shift 3: compressor C-7, bearing temperature reached 89 C.\\\\nDay 21, shift 2: fan F-12, filter replaced during scheduled stop.\\\\nDay 22, shift 3: pump P-101, alignment checked, offset 0.14 mm.\\\\nDay 23, shift 1: mixer M-5, motor current 3 A above baseline.\\\\nDay 24, shift 2: compressor C-7, alignment checked, offset 0.06 mm.\\\\nDay 25, shift 2: boiler B-2, operator reported unusual noise.\\\\nDay 26, shift 1: mixer M-5, oil level checked and topped up.\\\\nDay 27, shift 3: fan F-12, motor current 4 A above baseline.\\\\nDay 28, shift 1: chiller CH-4, bearing temperature reached 90 C.\\\\nDay 29, shift 1: chiller CH-4, seal leak observed, 4 drops per minute.\\\\nDay 30, shift 1: press PR-9, bearing temperature reached 72 C.\\\\nDay 31, shift 3: compressor C-7, operator reported unusual noise.\\\\nDay 32, shift 2: boiler B-2, seal leak observed, 30 drops per minute.\\\\nDay 33, shift 1: mixer M-5, filter replaced during scheduled stop.\\\\nDay 34, shift 1: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 35, shift 2: fan F-12, bearing temperature reached 71 C.\\\\nDay 36, shift 3: mixer M-5, vibration rose to 9.9 mm/s.\\\\n\\\\nSummarise the main trends in this log.\\"}\\n{\\"id\\": \\"s24\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Describe the main parts of a hydraulic system.\\"}\\n{\\"id\\": \\"l04\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 4:\\\\nDay 1, shift 2: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 2, shift 3: conveyor CV-3, alignment checked, offset 0.19 mm.\\\\nDay 3, shift 1: compressor C-7, seal leak observed, 26 drops per minute.\\\\nDay 4, shift 3: pump P-101, seal leak observed, 25 drops per minute.\\\\nDay 5, shift 2: boiler B-2, oil level checked and topped up.\\\\nDay 6, shift 3: chiller CH-4, oil level checked and topped up.\\\\nDay 7, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 8, shift 1: fan F-12, operator reported unusual noise.\\\\nDay 9, shift 3: compressor C-7, filter replaced during scheduled stop.\\\\nDay 10, shift 2: mixer M-5, bearing temperature reached 80 C.\\\\nDay 11, shift 2: chiller CH-4, operator reported unusual noise.\\\\nDay 12, shift 3: chiller CH-4, seal leak observed, 3 drops per minute.\\\\nDay 13, shift 1: pump P-101, vibration rose to 2.9 mm/s.\\\\nDay 14, shift 2: chiller CH-4, operator reported unusual noise.\\\\nDay 15, shift 2: pump P-101, motor current 9 A above baseline.\\\\nDay 16, shift 2: press PR-9, bearing temperature reached 83 C.\\\\nDay 17, shift 1: pump P-101, oil level checked and topped up.\\\\nDay 18, shift 1: conveyor CV-3, motor current 15 A above baseline.\\\\nDay 19, shift 2: boiler B-2, bearing temperature reached 76 C.\\\\nDay 20, shift 1: compressor C-7, filter replaced during scheduled stop.\\\\nDay 21, shift 1: compressor C-7, vibration rose to 4.5 mm/s.\\\\nDay 22, shift 3: compressor C-7, seal leak observed, 23 drops per minute.\\\\nDay 23, shift 3: fan F-12, seal leak observed, 11 drops per minute.\\\\nDay 24, shift 2: conveyor CV-3, bearing temperature reached 77 C.\\\\nDay 25, shift 1: mixer M-5, alignment checked, offset 0.19 mm.\\\\nDay 26, shift 1: boiler B-2, seal leak observed, 22 drops per minute.\\\\nDay 27, shift 2: chiller CH-4, seal leak observed, 29 drops per minute.\\\\nDay 28, shift 3: compressor C-7, alignment checked, offset 0.4 mm.\\\\nDay 29, shift 2: fan F-12, alignment checked, offset 0.36 mm.\\\\nDay 30, shift 2: fan F-12, motor current 7 A above baseline.\\\\nDay 31, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 32, shift 1: fan F-12, bearing temperature reached 71 C.\\\\nDay 33, shift 1: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 34, shift 3: conveyor CV-3, seal leak observed, 10 drops per minute.\\\\nDay 35, shift 2: fan F-12, filter replaced during scheduled stop.\\\\nDay 36, shift 2: conveyor CV-3, bearing temperature reached 56 C.\\\\n\\\\nWhich asset is most at risk of failure and why?\\"}\\n{\\"id\\": \\"s46\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a reliability centered maintenance program?\\"}\\n{\\"id\\": \\"l09\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 9:\\\\nDay 1, shift 3: boiler B-2, bearing temperature reached 81 C.\\\\nDay 2, shift 2: fan F-12, seal leak observed, 2 drops per minute.\\\\nDay 3, shift 1: pump P-101, operator reported unusual noise.\\\\nDay 4, shift 1: pump P-101, motor current 12 A above baseline.\\\\nDay 5, shift 1: press PR-9, filter replaced during scheduled stop.\\\\nDay 6, shift 3: conveyor CV-3, operator reported unusual noise.\\\\nDay 7, shift 3: chiller CH-4, bearing temperature reached 86 C.\\\\nDay 8, shift 2: compressor C-7, vibration rose to 5.0 mm/s.\\\\nDay 9, shift 1: boiler B-2, bearing temperature reached 59 C.\\\\nDay 10, shift 2: boiler B-2, oil level checked and topped up.\\\\nDay 11, shift 3: chiller CH-4, oil level checked and topped up.\\\\nDay 12, shift 3: fan F-12, seal leak observed, 25 drops per minute.\\\\nDay 13, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 14, shift 3: press PR-9, motor current 9 A above baseline.\\\\nDay 15, shift 2: pump P-101, filter replaced during scheduled stop.\\\\nDay 16, shift 3: conveyor CV-3, bearing temperature reached 71 C.\\\\nDay 17, shift 1: fan F-12, oil level checked and topped up.\\\\nDay 18, shift 2: mixer M-5, alignment checked, offset 0.32 mm.\\\\nDay 19, shift 1: press PR-9, vibration rose to 6.0 mm/s.\\\\nDay 20, shift 1: conveyor CV-3, bearing temperature reached 76 C.\\\\nDay 21, shift 3: press PR-9, oil level checked and topped up.\\\\nDay 22, shift 2: mixer M-5, operator reported unusual noise.\\\\nDay 23, shift 3: pump P-101, seal leak observed, 6 drops per minute.\\\\nDay 24, shift 3: boiler B-2, vibration rose to 8.6 mm/s.\\\\nDay 25, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 26, shift 3: fan F-12, motor current 14 A above baseline.\\\\nDay 27, shift 1: fan F-12, bearing temperature reached 82 C.\\\\nDay 28, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 29, shift 1: mixer M-5, vibration rose to 2.5 mm/s.\\\\nDay 30, shift 1: fan F-12, alignment checked, offset 0.08 mm.\\\\nDay 31, shift 1: fan F-12, seal leak observed, 18 drops per minute.\\\\nDay 32, shift 2: compressor C-7, motor current 11 A above baseline.\\\\nDay 33, shift 3: press PR-9, alignment checked, offset 0.21 mm.\\\\nDay 34, shift 3: compressor C-7, oil level checked and topped up.\\\\nDay 35, shift 2: pump P-101, alignment checked, offset 0.46 mm.\\\\nDay 36, shift 3: chiller CH-4, bearing temperature reached 61 C.\\\\n\\\\nSummarise the main trends in this log.\\"}\\n{\\"id\\": \\"s23\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the purpose of a check valve?\\"}\\n{\\"id\\": \\"s10\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"How does a variable frequency drive save energy?\\"}\\n{\\"id\\": \\"s11\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the role of a PLC in a factory?\\"}\\n"], "benchmark/accuracy/run_gsm8k.sh": ["67b262aa1074bacae97c9cc9bba97671f3d4b411d10c3c349b1c37a8f72d0dbb", "#!/usr/bin/env bash\\n# GSM8K accuracy against any OpenAI-compatible chat endpoint (llama.cpp or vLLM).\\n# Usage: run_gsm8k.sh BASE_URL MODEL OUT_DIR [LIMIT] [NUM_CONCURRENT]\\n#   BASE_URL  e.g. http://127.0.0.1:8080/v1/chat/completions\\n#   MODEL     model name the server expects (vLLM: --served-model-name; llama.cpp: anything)\\n#   LIMIT     first N test items (default 250, the fixed Phase 2 subset)\\n# Needs lm_eval on PATH (pip install -r benchmark/requirements.txt).\\nset -euo pipefail\\nBASE_URL=$1 MODEL=$2 OUT_DIR=$3 LIMIT=${4:-250} NUM_CONCURRENT=${5:-2}\\nHERE=$(cd \\"$(dirname \\"$0\\")\\" && pwd)\\nmkdir -p \\"$OUT_DIR\\"\\nlm_eval run \\\\\\n  --model local-chat-completions \\\\\\n  --model_args \\"model=$MODEL,base_url=$BASE_URL,num_concurrent=$NUM_CONCURRENT,max_retries=3,timeout=600,tokenizer_backend=None\\" \\\\\\n  --include_path \\"$HERE/tasks\\" \\\\\\n  --tasks gsm8k_pinned \\\\\\n  --apply_chat_template \\\\\\n  --fewshot_as_multiturn \\\\\\n  --limit \\"$LIMIT\\" \\\\\\n  --seed 0,1234,1234,1234 \\\\\\n  --log_samples \\\\\\n  --output_path \\"$OUT_DIR\\" 2>&1 | tee \\"$OUT_DIR/lm_eval.log\\"\\n"], "benchmark/accuracy/tasks/gsm8k_pinned.yaml": ["8fcecb0b938ca1f34db1e2ab4efdc5b75383253d9a2739dee9180c26a65ea2f5", "# Copy of lm-eval 0.4.13 tasks/gsm8k/gsm8k.yaml with the dataset revision pinned.\\ntask: gsm8k_pinned\\ndataset_path: openai/gsm8k\\ndataset_name: main\\ndataset_kwargs:\\n  revision: 740312add88f781978c0658806c59bc2815b9866\\noutput_type: generate_until\\ntraining_split: train\\nfewshot_split: train\\ntest_split: test\\ndoc_to_text: \\"Question: {{question}}\\\\nAnswer:\\"\\ndoc_to_target: \\"{{answer}}\\"\\nmetric_list:\\n  - metric: exact_match\\n    aggregation: mean\\n    higher_is_better: true\\n    ignore_case: true\\n    ignore_punctuation: false\\n    regexes_to_ignore:\\n      - \\",\\"\\n      - \\"\\\\\\\\$\\"\\n      - \\"(?s).*#### \\"\\n      - \\"\\\\\\\\.$\\"\\ngeneration_kwargs:\\n  until:\\n    - \\"Question:\\"\\n    - \\"</s>\\"\\n    - \\"<|im_end|>\\"\\n  do_sample: false\\n  temperature: 0.0\\nrepeats: 1\\nnum_fewshot: 5\\nfilter_list:\\n  - name: \\"strict-match\\"\\n    filter:\\n      - function: \\"regex\\"\\n        regex_pattern: \\"#### (\\\\\\\\-?[0-9\\\\\\\\.\\\\\\\\,]+)\\"\\n      - function: \\"take_first\\"\\n  - name: \\"flexible-extract\\"\\n    filter:\\n      - function: \\"regex\\"\\n        group_select: -1\\n        regex_pattern: \\"(-?[$0-9.,]{2,})|(-?[0-9]+)\\"\\n      - function: \\"take_first\\"\\nmetadata:\\n  version: 3.0\\n"], "benchmark/requirements.txt": ["c91bc10ea14e9da3c2e3124076c3d64827a21990af64e6de52e781bbe1744a60", "# This file was autogenerated by uv via the following command:\\n#    uv pip compile --universal --python-version 3.12 benchmark/requirements.in -o benchmark/requirements.txt\\nabsl-py==2.5.0\\n    # via rouge-score\\naiohappyeyeballs==2.7.1\\n    # via aiohttp\\naiohttp==3.14.4\\n    # via\\n    #   fsspec\\n    #   lm-eval\\naiosignal==1.4.0\\n    # via aiohttp\\nanyio==4.15.1\\n    # via\\n    #   httpx\\n    #   httpx2\\nattrs==26.1.0\\n    # via aiohttp\\ncertifi==2026.7.22\\n    # via\\n    #   httpcore\\n    #   httpx\\n    #   requests\\nchardet==6.0.0.post1\\n    # via mbstrdecoder\\ncharset-normalizer==3.5.2\\n    # via requests\\nclick==8.5.0\\n    # via\\n    #   huggingface-hub\\n    #   nltk\\ncloudpickle==3.1.2\\n    # via joblib\\ncolorama==0.4.6\\n    # via\\n    #   pytest\\n    #   sacrebleu\\n    #   tqdm\\ncontourpy==1.4.0\\n    # via matplotlib\\ncycler==0.12.1\\n    # via matplotlib\\ndataproperty==1.1.1\\n    # via\\n    #   pytablewriter\\n    #   tabledata\\ndatasets==5.1.0\\n    # via\\n    #   evaluate\\n    #   lm-eval\\ndefusedxml==0.7.1\\n    # via nltk\\ndill==0.4.1\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   lm-eval\\n    #   multiprocess\\nevaluate==0.4.6\\n    # via lm-eval\\nfilelock==4.0.12\\n    # via\\n    #   datasets\\n    #   huggingface-hub\\nfonttools==4.66.1\\n    # via matplotlib\\nfrozenlist==1.8.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\nfsspec==2026.7.0\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   huggingface-hub\\nh11==0.16.0\\n    # via\\n    #   httpcore\\n    #   httpcore2\\nhf-xet==1.7.0 ; platform_machine == \'AMD64\' or platform_machine == \'aarch64\' or platform_machine == \'amd64\' or platform_machine == \'arm64\' or platform_machine == \'x86_64\'\\n    # via huggingface-hub\\nhttpcore==1.0.9\\n    # via httpx\\nhttpcore2==2.13.1 ; sys_platform != \'emscripten\'\\n    # via httpx2\\nhttpx==0.28.1\\n    # via -r benchmark/requirements.in\\nhttpx2==2.13.1\\n    # via huggingface-hub\\nhttpx2-jsfetch==1.0 ; sys_platform == \'emscripten\'\\n    # via httpx2\\nhuggingface-hub==2.1.1\\n    # via\\n    #   datasets\\n    #   evaluate\\nidna==3.20\\n    # via\\n    #   anyio\\n    #   httpx\\n    #   httpx2\\n    #   requests\\n    #   yarl\\niniconfig==2.3.1\\n    # via pytest\\njinja2==3.1.6\\n    # via lm-eval\\njoblib==1.6.0\\n    # via\\n    #   nltk\\n    #   scikit-learn\\nkiwisolver==1.5.1\\n    # via matplotlib\\nlm-eval==0.4.13\\n    # via -r benchmark/requirements.in\\nlxml==6.1.3\\n    # via sacrebleu\\nmarkupsafe==3.0.4\\n    # via jinja2\\nmatplotlib==3.11.2\\n    # via -r benchmark/requirements.in\\nmbstrdecoder==1.1.5\\n    # via\\n    #   dataproperty\\n    #   pytablewriter\\n    #   typepy\\nmore-itertools==11.1.0\\n    # via lm-eval\\nmultidict==6.9.1\\n    # via\\n    #   aiohttp\\n    #   yarl\\nmultiprocess==0.70.19\\n    # via\\n    #   datasets\\n    #   evaluate\\nnarwhals==2.26.0\\n    # via scikit-learn\\nnltk==3.10.3\\n    # via rouge-score\\nnumpy==2.5.3\\n    # via\\n    #   contourpy\\n    #   datasets\\n    #   evaluate\\n    #   lm-eval\\n    #   matplotlib\\n    #   pandas\\n    #   rouge-score\\n    #   sacrebleu\\n    #   scikit-learn\\n    #   scipy\\npackaging==26.3\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   huggingface-hub\\n    #   matplotlib\\n    #   pytest\\n    #   typepy\\npandas==3.0.6\\n    # via\\n    #   datasets\\n    #   evaluate\\npathvalidate==3.3.1\\n    # via pytablewriter\\npillow==12.3.0\\n    # via matplotlib\\npluggy==1.6.0\\n    # via pytest\\nportalocker==4.4.0\\n    # via sacrebleu\\npropcache==0.5.4\\n    # via\\n    #   aiohttp\\n    #   yarl\\npyarrow==25.0.1\\n    # via datasets\\npygments==2.21.0\\n    # via pytest\\npyparsing==3.3.3\\n    # via matplotlib\\npytablewriter==1.2.1\\n    # via lm-eval\\npytest==9.1.1\\n    # via -r benchmark/requirements.in\\npython-dateutil==2.9.0.post0\\n    # via\\n    #   matplotlib\\n    #   pandas\\n    #   typepy\\npytz==2026.5\\n    # via typepy\\npyyaml==6.0.3\\n    # via\\n    #   datasets\\n    #   huggingface-hub\\nregex==2026.9.29\\n    # via\\n    #   nltk\\n    #   sacrebleu\\n    #   tiktoken\\nrequests==2.34.2\\n    # via\\n    #   evaluate\\n    #   lm-eval\\n    #   tiktoken\\nrouge-score==0.1.2\\n    # via lm-eval\\nsacrebleu==2.6.0\\n    # via lm-eval\\nscikit-learn==1.9.1\\n    # via lm-eval\\nscipy==1.18.1\\n    # via scikit-learn\\nsetuptools==84.0.0\\n    # via pytablewriter\\nsix==1.17.0\\n    # via\\n    #   python-dateutil\\n    #   rouge-score\\nsqlitedict==2.1.0\\n    # via lm-eval\\ntabledata==1.3.5\\n    # via pytablewriter\\ntabulate==0.10.0\\n    # via sacrebleu\\ntcolorpy==0.1.7\\n    # via pytablewriter\\ntenacity==9.2.1\\n    # via lm-eval\\nthreadpoolctl==3.7.0\\n    # via scikit-learn\\ntiktoken==0.14.0\\n    # via lm-eval\\ntqdm==4.70.1\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   huggingface-hub\\n    #   lm-eval\\n    #   nltk\\ntruststore==0.10.4 ; sys_platform != \'emscripten\'\\n    # via\\n    #   httpcore2\\n    #   httpx2\\ntypepy==1.3.5\\n    # via\\n    #   dataproperty\\n    #   pytablewriter\\n    #   tabledata\\ntyping-extensions==4.16.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\n    #   anyio\\n    #   httpx2\\n    #   huggingface-hub\\n    #   lm-eval\\ntzdata==2026.5 ; sys_platform == \'emscripten\' or sys_platform == \'win32\'\\n    # via pandas\\nurllib3==2.8.0\\n    # via requests\\nword2number==1.1\\n    # via lm-eval\\nxxhash==4.0.1\\n    # via\\n    #   datasets\\n    #   evaluate\\nyarl==1.25.1\\n    # via aiohttp\\n"], "notebooks/vllm-requirements.txt": ["9218775e0b0c2b1a0675402c53a77297525f0d2675d18548f2db5a83841bfe7e", "# Generated from notebooks/vllm-requirements.in (see the commands there). Install with --torch-backend cu129.\\nagent-detector==2.0.0\\n    # via fastapi-cloud-cli\\naiohappyeyeballs==2.7.1\\n    # via aiohttp\\naiohttp==3.14.4\\n    # via vllm\\naiosignal==1.4.0\\n    # via aiohttp\\nannotated-doc==0.0.5\\n    # via\\n    #   fastapi\\n    #   typer\\nannotated-types==0.8.0\\n    # via pydantic\\nanthropic==1.11.0\\n    # via vllm\\nanyio==4.15.1\\n    # via\\n    #   anthropic\\n    #   httpx\\n    #   httpx2\\n    #   mcp\\n    #   openai\\n    #   sse-starlette\\n    #   starlette\\n    #   watchfiles\\napache-tvm-ffi==0.1.11\\n    # via\\n    #   flashinfer-python\\n    #   nvidia-cudnn-frontend\\n    #   quack-kernels\\n    #   tilelang\\n    #   tokenspeed-mla\\n    #   vllm\\n    #   xgrammar\\nastor==0.8.1\\n    # via depyf\\nattrs==26.1.0\\n    # via\\n    #   aiohttp\\n    #   jsonschema\\n    #   referencing\\nblake3==1.0.10\\n    # via vllm\\ncachetools==7.2.1\\n    # via vllm\\ncbor2==6.1.5\\n    # via vllm\\ncertifi==2026.7.22\\n    # via\\n    #   httpcore\\n    #   httpx\\n    #   requests\\n    #   sentry-sdk\\ncffi==2.1.1\\n    # via cryptography\\ncharset-normalizer==3.5.2\\n    # via requests\\nclick==8.5.0\\n    # via\\n    #   flashinfer-python\\n    #   huggingface-hub\\n    #   rich-toolkit\\n    #   uvicorn\\ncloudpickle==3.1.2\\n    # via\\n    #   tilelang\\n    #   vllm\\ncompressed-tensors==0.17.0\\n    # via vllm\\ncryptography==50.0.2\\n    # via pyjwt\\ncuda-bindings==12.9.9\\n    # via\\n    #   cuda-python\\n    #   humming-kernels\\n    #   torch\\ncuda-core==1.2.1\\n    # via\\n    #   nccl-extensions\\n    #   nccl4py\\ncuda-pathfinder==1.8.3\\n    # via\\n    #   cuda-bindings\\n    #   cuda-core\\n    #   nccl-extensions\\n    #   nccl4py\\ncuda-python==12.9.7\\n    # via\\n    #   flashinfer-python\\n    #   nvidia-cutlass-dsl-libs-base\\n    #   nvidia-cutlass-dsl-libs-core\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\ncuda-tile==1.6.0\\n    # via flashinfer-python\\ncuda-toolkit==12.9.1\\n    # via torch\\ndepyf==0.20.0\\n    # via vllm\\ndetect-installer==0.2.1\\n    # via fastapi-cloud-cli\\ndill==0.4.1\\n    # via depyf\\ndnspython==2.8.0\\n    # via email-validator\\ndocstring-parser==0.18.0\\n    # via anthropic\\neinops==0.8.2\\n    # via\\n    #   flashinfer-python\\n    #   quack-kernels\\n    #   vllm\\nemail-validator==2.3.0\\n    # via\\n    #   fastapi\\n    #   pydantic\\nfastapi==0.136.3\\n    # via\\n    #   model-hosting-container-standards\\n    #   vllm\\nfastapi-cli==0.0.32\\n    # via fastapi\\nfastapi-cloud-cli==0.26.0\\n    # via fastapi-cli\\nfastar==0.12.0\\n    # via\\n    #   fastapi\\n    #   fastapi-cloud-cli\\nfastsafetensors==0.4.0\\n    # via vllm\\nfilelock==4.0.12\\n    # via\\n    #   huggingface-hub\\n    #   torch\\n    #   vllm\\nflashinfer-python==0.7.0.post1\\n    # via vllm\\nfrozenlist==1.8.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\nfsspec==2026.9.0\\n    # via\\n    #   huggingface-hub\\n    #   torch\\ngoogleapis-common-protos==1.75.5\\n    # via\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\ngrpcio==1.84.0\\n    # via opentelemetry-exporter-otlp-proto-grpc\\nh11==0.16.0\\n    # via\\n    #   httpcore\\n    #   httpcore2\\n    #   uvicorn\\nhf-xet==1.7.0\\n    # via huggingface-hub\\nhttpcore==1.0.9\\n    # via httpx\\nhttpcore2==2.13.1\\n    # via httpx2\\nhttptools==0.8.0\\n    # via uvicorn\\nhttpx==0.28.1\\n    # via\\n    #   fastapi\\n    #   fastapi-cloud-cli\\n    #   huggingface-hub\\n    #   model-hosting-container-standards\\nhttpx2==2.13.1\\n    # via\\n    #   anthropic\\n    #   mcp\\n    #   openai\\nhuggingface-hub==1.33.0\\n    # via\\n    #   tokenizers\\n    #   transformers\\n    #   vllm\\nhumming-kernels==0.1.16\\n    # via\\n    #   --override notebooks/vllm-overrides.txt\\n    #   vllm\\nidna==3.20\\n    # via\\n    #   anyio\\n    #   email-validator\\n    #   httpx\\n    #   httpx2\\n    #   requests\\n    #   yarl\\nijson==3.5.1\\n    # via vllm\\ninstanttensor==0.2.1\\n    # via vllm\\ninteregular==0.3.3\\n    # via lm-format-enforcer\\njinja2==3.1.6\\n    # via\\n    #   fastapi\\n    #   humming-kernels\\n    #   torch\\njiter==0.17.0\\n    # via\\n    #   anthropic\\n    #   openai\\njmespath==1.1.0\\n    # via model-hosting-container-standards\\njsonschema==4.26.0\\n    # via\\n    #   mcp\\n    #   mistral-common\\n    #   vllm\\njsonschema-specifications==2025.9.1\\n    # via jsonschema\\nlark==1.2.2\\n    # via vllm\\nllguidance==1.7.6\\n    # via vllm\\nllvmlite==0.47.0\\n    # via numba\\nlm-format-enforcer==0.11.3\\n    # via vllm\\nloguru==0.7.3\\n    # via compressed-tensors\\nmarkdown-it-py==4.2.0\\n    # via rich\\nmarkupsafe==3.0.4\\n    # via jinja2\\nmcp==2.3.0\\n    # via vllm\\nmcp-types==2.3.0\\n    # via mcp\\nmdurl==0.1.2\\n    # via markdown-it-py\\nmistral-common==1.12.0\\n    # via vllm\\nml-dtypes==0.6.0\\n    # via tilelang\\nmodel-hosting-container-standards==0.1.16\\n    # via vllm\\nmpmath==1.3.0\\n    # via sympy\\nmsgspec==0.22.0\\n    # via vllm\\nmultidict==6.9.1\\n    # via\\n    #   aiohttp\\n    #   yarl\\nnccl-extensions==0.1.0\\n    # via flashinfer-python\\nnccl4py==0.6.0\\n    # via\\n    #   flashinfer-python\\n    #   nccl-extensions\\nnetworkx==3.7\\n    # via torch\\nninja==1.13.2\\n    # via\\n    #   flashinfer-python\\n    #   vllm\\nnumba==0.65.0\\n    # via vllm\\nnumpy==2.3.5\\n    # via\\n    #   cuda-core\\n    #   flashinfer-python\\n    #   humming-kernels\\n    #   mistral-common\\n    #   ml-dtypes\\n    #   nccl-extensions\\n    #   nccl4py\\n    #   numba\\n    #   nvidia-cutlass-dsl-libs-base\\n    #   nvidia-cutlass-dsl-libs-core\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\n    #   opencv-python-headless\\n    #   tilelang\\n    #   torchvision\\n    #   transformers\\n    #   vllm\\n    #   xgrammar\\nnvidia-cublas-cu12==12.9.1.4\\n    # via\\n    #   cuda-toolkit\\n    #   nvidia-cudnn-cu12\\n    #   nvidia-cusolver-cu12\\nnvidia-cuda-cccl-cu12==12.9.27\\n    # via humming-kernels\\nnvidia-cuda-cupti-cu12==12.9.79\\n    # via cuda-toolkit\\nnvidia-cuda-nvcc-cu12==12.9.86\\n    # via humming-kernels\\nnvidia-cuda-nvdisasm==13.4.92\\n    # via\\n    #   nvidia-cutlass-dsl-libs-base\\n    #   nvidia-cutlass-dsl-libs-core\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\nnvidia-cuda-nvrtc-cu12==12.9.86\\n    # via\\n    #   cuda-toolkit\\n    #   humming-kernels\\nnvidia-cuda-runtime-cu12==12.9.79\\n    # via\\n    #   cuda-toolkit\\n    #   humming-kernels\\nnvidia-cudnn-cu12==9.20.0.48\\n    # via torch\\nnvidia-cudnn-frontend==1.30.0\\n    # via\\n    #   flashinfer-python\\n    #   vllm\\nnvidia-cufft-cu12==11.4.1.4\\n    # via cuda-toolkit\\nnvidia-cufile-cu12==1.14.1.1\\n    # via cuda-toolkit\\nnvidia-curand-cu12==10.3.10.19\\n    # via cuda-toolkit\\nnvidia-cusolver-cu12==11.7.5.82\\n    # via cuda-toolkit\\nnvidia-cusparse-cu12==12.5.10.65\\n    # via\\n    #   cuda-toolkit\\n    #   nvidia-cusolver-cu12\\nnvidia-cusparselt-cu12==0.8.1\\n    # via torch\\nnvidia-cutlass-dsl==4.7.1\\n    # via\\n    #   flashinfer-python\\n    #   nvidia-cudnn-frontend\\n    #   quack-kernels\\n    #   tokenspeed-mla\\n    #   vllm\\nnvidia-cutlass-dsl-libs-base==4.7.1\\n    # via\\n    #   nvidia-cutlass-dsl\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\nnvidia-cutlass-dsl-libs-core==4.7.1\\n    # via nvidia-cutlass-dsl-libs-base\\nnvidia-cutlass-dsl-libs-cu12==4.7.1\\n    # via nvidia-cutlass-dsl\\nnvidia-cutlass-dsl-libs-cu13==4.7.1\\n    # via nvidia-cutlass-dsl\\nnvidia-ml-py==13.615.71\\n    # via flashinfer-python\\nnvidia-nccl-cu12==2.29.7\\n    # via torch\\nnvidia-nvjitlink-cu12==12.9.86\\n    # via\\n    #   cuda-toolkit\\n    #   nvidia-cufft-cu12\\n    #   nvidia-cusolver-cu12\\n    #   nvidia-cusparse-cu12\\nnvidia-nvshmem-cu12==3.4.5\\n    # via torch\\nnvidia-nvtx-cu12==12.9.79\\n    # via cuda-toolkit\\nnvtx==0.2.15\\n    # via vllm\\nopenai==3.26.0\\n    # via vllm\\nopencv-python-headless==5.0.0.93\\n    # via\\n    #   mistral-common\\n    #   vllm\\nopentelemetry-api==1.45.1\\n    # via\\n    #   mcp\\n    #   opentelemetry-exporter-http-transport\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\n    #   opentelemetry-sdk\\n    #   opentelemetry-semantic-conventions\\n    #   vllm\\nopentelemetry-exporter-http-transport==0.66b1\\n    # via opentelemetry-exporter-otlp-proto-http\\nopentelemetry-exporter-otlp==1.45.1\\n    # via vllm\\nopentelemetry-exporter-otlp-common==0.66b1\\n    # via\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\nopentelemetry-exporter-otlp-proto-common==1.45.1\\n    # via\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\nopentelemetry-exporter-otlp-proto-grpc==1.45.1\\n    # via opentelemetry-exporter-otlp\\nopentelemetry-exporter-otlp-proto-http==1.45.1\\n    # via opentelemetry-exporter-otlp\\nopentelemetry-proto==1.45.1\\n    # via\\n    #   opentelemetry-exporter-otlp-proto-common\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\nopentelemetry-sdk==1.45.1\\n    # via\\n    #   opentelemetry-exporter-otlp-common\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\n    #   opentelemetry-semantic-conventions-ai\\n    #   vllm\\nopentelemetry-semantic-conventions==0.66b1\\n    # via\\n    #   opentelemetry-sdk\\n    #   opentelemetry-semantic-conventions-ai\\nopentelemetry-semantic-conventions-ai==0.5.1\\n    # via vllm\\noss-harmony==0.0.11\\n    # via vllm\\noutlines-core==0.2.14\\n    # via vllm\\npackaging==26.3\\n    # via\\n    #   flashinfer-python\\n    #   huggingface-hub\\n    #   lm-format-enforcer\\n    #   nccl-extensions\\n    #   nccl4py\\n    #   transformers\\npartial-json-parser==0.2.1.1.post7\\n    # via vllm\\npillow==12.3.0\\n    # via\\n    #   mistral-common\\n    #   torchvision\\n    #   vllm\\nprometheus-client==0.26.0\\n    # via\\n    #   prometheus-fastapi-instrumentator\\n    #   vllm\\nprometheus-fastapi-instrumentator==8.1.0\\n    # via vllm\\npropcache==0.5.4\\n    # via\\n    #   aiohttp\\n    #   yarl\\nprotobuf==7.36.2\\n    # via\\n    #   googleapis-common-protos\\n    #   nvidia-cutlass-dsl-libs-base\\n    #   nvidia-cutlass-dsl-libs-core\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\n    #   opentelemetry-proto\\n    #   vllm\\npsutil==7.2.2\\n    # via\\n    #   tilelang\\n    #   vllm\\npy-cpuinfo==9.0.0\\n    # via vllm\\npybase64==1.5.1\\n    # via vllm\\npycountry==26.2.16\\n    # via pydantic-extra-types\\npycparser==3.0\\n    # via cffi\\npydantic==2.13.5\\n    # via\\n    #   anthropic\\n    #   compressed-tensors\\n    #   fastapi\\n    #   fastapi-cloud-cli\\n    #   lm-format-enforcer\\n    #   mcp\\n    #   mcp-types\\n    #   mistral-common\\n    #   model-hosting-container-standards\\n    #   openai\\n    #   oss-harmony\\n    #   pydantic-extra-types\\n    #   pydantic-settings\\n    #   vllm\\n    #   xgrammar\\npydantic-core==2.46.5\\n    # via pydantic\\npydantic-extra-types==2.11.1\\n    # via\\n    #   fastapi\\n    #   mistral-common\\npydantic-settings==2.15.0\\n    # via fastapi\\npygments==2.21.0\\n    # via rich\\npyjwt==2.15.1\\n    # via mcp\\npynvvideocodec==2.0.4\\n    # via vllm\\npython-dotenv==1.2.4\\n    # via\\n    #   pydantic-settings\\n    #   uvicorn\\npython-json-logger==4.2.0\\n    # via vllm\\npython-multipart==0.0.32\\n    # via\\n    #   fastapi\\n    #   mcp\\npyyaml==6.0.3\\n    # via\\n    #   huggingface-hub\\n    #   lm-format-enforcer\\n    #   transformers\\n    #   uvicorn\\n    #   vllm\\npyzmq==27.2.0\\n    # via vllm\\nquack-kernels==0.6.5\\n    # via vllm\\nreferencing==0.37.0\\n    # via\\n    #   jsonschema\\n    #   jsonschema-specifications\\nregex==2026.9.29\\n    # via\\n    #   tiktoken\\n    #   transformers\\n    #   vllm\\nrequests==2.34.2\\n    # via\\n    #   flashinfer-python\\n    #   mistral-common\\n    #   opentelemetry-exporter-http-transport\\n    #   opentelemetry-exporter-otlp-proto-http\\n    #   tiktoken\\n    #   vllm\\nrich==15.0.0\\n    # via\\n    #   rich-toolkit\\n    #   typer\\nrich-toolkit==0.20.5\\n    # via\\n    #   fastapi-cli\\n    #   fastapi-cloud-cli\\nrignore==0.8.1\\n    # via fastapi-cloud-cli\\nrpds-py==2026.9.1\\n    # via\\n    #   jsonschema\\n    #   referencing\\nsafetensors==0.8.0\\n    # via\\n    #   humming-kernels\\n    #   transformers\\n    #   vllm\\nsentencepiece==0.2.2\\n    # via vllm\\nsentry-sdk==2.71.0\\n    # via fastapi-cloud-cli\\nsetproctitle==1.3.8\\n    # via vllm\\nsetuptools==80.10.2\\n    # via\\n    #   model-hosting-container-standards\\n    #   torch\\n    #   vllm\\nshellingham==1.5.4\\n    # via typer\\nsix==1.17.0\\n    # via vllm\\nsniffio==1.3.1\\n    # via\\n    #   anthropic\\n    #   openai\\nsse-starlette==3.5.0\\n    # via mcp\\nstarlette==1.7.0\\n    # via\\n    #   fastapi\\n    #   mcp\\n    #   model-hosting-container-standards\\n    #   prometheus-fastapi-instrumentator\\n    #   sse-starlette\\n    #   vllm\\nsupervisor==4.3.0\\n    # via model-hosting-container-standards\\nsympy==1.14.0\\n    # via torch\\ntabulate==0.10.0\\n    # via\\n    #   flashinfer-python\\n    #   humming-kernels\\ntiktoken==0.14.0\\n    # via\\n    #   mistral-common\\n    #   vllm\\ntilelang==0.1.12\\n    # via vllm\\ntokenizers==0.23.2\\n    # via\\n    #   transformers\\n    #   vllm\\ntokenspeed-mla==0.1.8\\n    # via vllm\\ntokenspeed-triton==3.8.10.post20260920\\n    # via tokenspeed-mla\\ntorch==2.13.0+cu129\\n    # via\\n    #   compressed-tensors\\n    #   flashinfer-python\\n    #   humming-kernels\\n    #   instanttensor\\n    #   quack-kernels\\n    #   tilelang\\n    #   tokenspeed-mla\\n    #   torch-c-dlpack-ext\\n    #   torchvision\\n    #   vllm\\n    #   xgrammar\\ntorch-c-dlpack-ext==0.1.5\\n    # via\\n    #   quack-kernels\\n    #   tilelang\\ntorchaudio==2.11.0+cu129\\n    # via vllm\\ntorchcodec==0.16.0+cu129\\n    # via vllm\\ntorchvision==0.28.0+cu129\\n    # via vllm\\ntqdm==4.70.1\\n    # via\\n    #   flashinfer-python\\n    #   huggingface-hub\\n    #   humming-kernels\\n    #   tilelang\\n    #   transformers\\n    #   vllm\\ntransformers==5.17.0\\n    # via\\n    #   compressed-tensors\\n    #   vllm\\n    #   xgrammar\\ntriton==3.7.1\\n    # via\\n    #   humming-kernels\\n    #   torch\\n    #   xgrammar\\ntruststore==0.10.4\\n    # via\\n    #   httpcore2\\n    #   httpx2\\ntyper==0.27.3\\n    # via\\n    #   fastapi-cli\\n    #   fastapi-cloud-cli\\n    #   fastsafetensors\\n    #   transformers\\ntyping-extensions==4.16.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\n    #   anthropic\\n    #   anyio\\n    #   apache-tvm-ffi\\n    #   cuda-tile\\n    #   fastapi\\n    #   grpcio\\n    #   httpx2\\n    #   huggingface-hub\\n    #   mcp\\n    #   mcp-types\\n    #   mistral-common\\n    #   nvidia-cutlass-dsl-libs-base\\n    #   nvidia-cutlass-dsl-libs-core\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\n    #   openai\\n    #   opentelemetry-api\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\n    #   opentelemetry-sdk\\n    #   opentelemetry-semantic-conventions\\n    #   pydantic\\n    #   pydantic-core\\n    #   pydantic-extra-types\\n    #   referencing\\n    #   rich-toolkit\\n    #   starlette\\n    #   tilelang\\n    #   torch\\n    #   typing-inspection\\n    #   vllm\\n    #   xgrammar\\ntyping-inspection==0.4.4\\n    # via\\n    #   fastapi\\n    #   mcp\\n    #   pydantic\\n    #   pydantic-settings\\nurllib3==2.8.0\\n    # via\\n    #   requests\\n    #   sentry-sdk\\nuvicorn==0.54.0\\n    # via\\n    #   fastapi\\n    #   fastapi-cli\\n    #   fastapi-cloud-cli\\n    #   mcp\\nuvloop==0.23.0\\n    # via uvicorn\\nvllm @ https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0%2Bcu129-cp38-abi3-manylinux_2_28_x86_64.whl\\n    # via -r notebooks/vllm-requirements.in\\nwatchfiles==1.3.0\\n    # via\\n    #   uvicorn\\n    #   vllm\\nwebsockets==17.2\\n    # via uvicorn\\nxgrammar==0.2.7\\n    # via vllm\\nyarl==1.25.1\\n    # via aiohttp\\nz3-solver==4.15.4.0\\n    # via tilelang\\n"], "notebooks/trtllm-requirements.txt": ["06b979189e6164aba866d8214711ac9575be405d1209827d42094bb4357aefec", "# This file was autogenerated by uv via the following command:\\n#    uv pip compile notebooks/trtllm-requirements.in -o notebooks/trtllm-requirements.txt --python-version 3.10 --python-platform x86_64-unknown-linux-gnu --torch-backend cu124 --index-strategy unsafe-best-match\\naccelerate==1.15.0\\n    # via tensorrt-llm\\naenum==3.1.17\\n    # via tensorrt-llm\\naiohappyeyeballs==2.7.1\\n    # via aiohttp\\naiohttp==3.14.4\\n    # via\\n    #   datasets\\n    #   fsspec\\naiosignal==1.4.0\\n    # via aiohttp\\nannotated-types==0.8.0\\n    # via pydantic\\nanyio==4.15.1\\n    # via\\n    #   httpx\\n    #   openai\\n    #   starlette\\nasync-timeout==5.0.1\\n    # via aiohttp\\nattrs==26.1.0\\n    # via aiohttp\\nbuild==1.6.1\\n    # via tensorrt-llm\\ncertifi==2026.7.22\\n    # via\\n    #   httpcore\\n    #   httpx\\n    #   requests\\ncharset-normalizer==3.5.2\\n    # via requests\\nclick==8.5.0\\n    # via\\n    #   click-option-group\\n    #   tensorrt-llm\\n    #   uvicorn\\nclick-option-group==0.5.9\\n    # via tensorrt-llm\\ncloudpickle==3.1.2\\n    # via nvidia-modelopt\\ncolored==2.3.2\\n    # via tensorrt-llm\\ncuda-python==12.5.0\\n    # via\\n    #   -r notebooks/trtllm-requirements.in\\n    #   tensorrt-llm\\ndatasets==2.14.4\\n    # via evaluate\\ndiffusers==0.39.0\\n    # via tensorrt-llm\\ndill==0.3.7\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   multiprocess\\ndistro==1.9.0\\n    # via openai\\nevaluate==0.4.6\\n    # via tensorrt-llm\\nexceptiongroup==1.3.1\\n    # via anyio\\nfastapi==0.112.0\\n    # via -r notebooks/trtllm-requirements.in\\nfilelock==4.0.12\\n    # via\\n    #   diffusers\\n    #   huggingface-hub\\n    #   torch\\n    #   transformers\\n    #   triton\\nfrozenlist==1.8.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\nfsspec==2026.9.0\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   huggingface-hub\\n    #   torch\\nh11==0.16.0\\n    # via\\n    #   httpcore\\n    #   uvicorn\\nh5py==3.10.0\\n    # via tensorrt-llm\\nhf-xet==1.7.0\\n    # via huggingface-hub\\nhttpcore==1.0.9\\n    # via httpx\\nhttpx==0.28.1\\n    # via\\n    #   diffusers\\n    #   openai\\nhuggingface-hub==0.36.2\\n    # via\\n    #   accelerate\\n    #   datasets\\n    #   diffusers\\n    #   evaluate\\n    #   optimum\\n    #   tokenizers\\n    #   transformers\\nidna==3.20\\n    # via\\n    #   anyio\\n    #   httpx\\n    #   requests\\n    #   yarl\\nimportlib-metadata==9.0.1\\n    # via\\n    #   build\\n    #   diffusers\\njinja2==3.1.6\\n    # via torch\\nlark==1.3.1\\n    # via tensorrt-llm\\nmarkdown-it-py==4.2.0\\n    # via rich\\nmarkupsafe==3.0.4\\n    # via jinja2\\nmdurl==0.1.2\\n    # via markdown-it-py\\nml-dtypes==0.5.4\\n    # via onnx\\nmpi4py==3.1.6\\n    # via\\n    #   -r notebooks/trtllm-requirements.in\\n    #   tensorrt-llm\\nmpmath==1.3.0\\n    # via\\n    #   sympy\\n    #   tensorrt-llm\\nmultidict==6.9.1\\n    # via\\n    #   aiohttp\\n    #   yarl\\nmultiprocess==0.70.15\\n    # via\\n    #   datasets\\n    #   evaluate\\nnetworkx==3.4.2\\n    # via torch\\nninja==1.13.2\\n    # via nvidia-modelopt\\nnumpy==1.26.4\\n    # via\\n    #   accelerate\\n    #   datasets\\n    #   diffusers\\n    #   evaluate\\n    #   h5py\\n    #   ml-dtypes\\n    #   nvidia-modelopt\\n    #   onnx\\n    #   optimum\\n    #   pandas\\n    #   scipy\\n    #   tensorrt-llm\\n    #   torchvision\\n    #   transformers\\nnvidia-cublas-cu12==12.4.2.65\\n    # via\\n    #   nvidia-cudnn-cu12\\n    #   nvidia-cusolver-cu12\\n    #   torch\\nnvidia-cuda-cupti-cu12==12.4.99\\n    # via torch\\nnvidia-cuda-nvrtc-cu12==12.4.99\\n    # via torch\\nnvidia-cuda-runtime-cu12==12.4.99\\n    # via torch\\nnvidia-cudnn-cu12==9.1.0.70\\n    # via torch\\nnvidia-cufft-cu12==11.2.0.44\\n    # via torch\\nnvidia-curand-cu12==10.3.5.119\\n    # via torch\\nnvidia-cusolver-cu12==11.6.0.99\\n    # via torch\\nnvidia-cusparse-cu12==12.3.0.142\\n    # via\\n    #   nvidia-cusolver-cu12\\n    #   torch\\nnvidia-modelopt==0.17.0\\n    # via tensorrt-llm\\nnvidia-nccl-cu12==2.20.5\\n    # via torch\\nnvidia-nvjitlink-cu12==12.4.99\\n    # via\\n    #   nvidia-cusolver-cu12\\n    #   nvidia-cusparse-cu12\\n    #   torch\\nnvidia-nvtx-cu12==12.4.99\\n    # via torch\\nonnx==1.23.2\\n    # via tensorrt-llm\\nopenai==1.39.0\\n    # via tensorrt-llm\\noptimum==2.3.0\\n    # via tensorrt-llm\\npackaging==26.3\\n    # via\\n    #   accelerate\\n    #   build\\n    #   datasets\\n    #   evaluate\\n    #   huggingface-hub\\n    #   nvidia-modelopt\\n    #   optimum\\n    #   transformers\\n    #   wheel\\npandas==2.3.3\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   tensorrt-llm\\npillow==10.3.0\\n    # via\\n    #   diffusers\\n    #   tensorrt-llm\\n    #   torchvision\\npolygraphy==0.53.6\\n    # via tensorrt-llm\\nprometheus-client==0.26.0\\n    # via -r notebooks/trtllm-requirements.in\\npropcache==0.5.4\\n    # via\\n    #   aiohttp\\n    #   yarl\\nprotobuf==7.36.2\\n    # via onnx\\npsutil==7.2.2\\n    # via\\n    #   accelerate\\n    #   tensorrt-llm\\npulp==3.3.2\\n    # via\\n    #   nvidia-modelopt\\n    #   tensorrt-llm\\npyarrow==20.0.0\\n    # via\\n    #   -r notebooks/trtllm-requirements.in\\n    #   datasets\\npydantic==2.14.0\\n    # via\\n    #   fastapi\\n    #   nvidia-modelopt\\n    #   openai\\n    #   tensorrt-llm\\npydantic-core==2.50.0\\n    # via pydantic\\npygments==2.21.0\\n    # via rich\\npynvml==11.5.3\\n    # via\\n    #   -r notebooks/trtllm-requirements.in\\n    #   tensorrt-llm\\npyproject-hooks==1.3.3\\n    # via build\\npython-dateutil==2.9.0.post0\\n    # via pandas\\npytz==2026.5\\n    # via pandas\\npyyaml==6.0.3\\n    # via\\n    #   accelerate\\n    #   datasets\\n    #   huggingface-hub\\n    #   transformers\\nregex==2026.9.29\\n    # via\\n    #   diffusers\\n    #   nvidia-modelopt\\n    #   transformers\\nrequests==2.34.2\\n    # via\\n    #   datasets\\n    #   diffusers\\n    #   evaluate\\n    #   huggingface-hub\\n    #   transformers\\nrich==15.0.0\\n    # via nvidia-modelopt\\nsafetensors==0.8.0\\n    # via\\n    #   accelerate\\n    #   diffusers\\n    #   nvidia-modelopt\\n    #   transformers\\nscipy==1.15.3\\n    # via nvidia-modelopt\\nsentencepiece==0.2.2\\n    # via tensorrt-llm\\nsix==1.17.0\\n    # via python-dateutil\\nsniffio==1.3.1\\n    # via openai\\nstarlette==0.37.2\\n    # via fastapi\\nstrenum==0.4.15\\n    # via tensorrt-llm\\nsympy==1.14.0\\n    # via torch\\ntensorrt==10.4.0\\n    # via tensorrt-llm\\ntensorrt-cu12==10.4.0\\n    # via tensorrt\\ntensorrt-llm==0.14.0\\n    # via -r notebooks/trtllm-requirements.in\\ntokenizers==0.20.3\\n    # via transformers\\ntomli==2.5.0\\n    # via build\\ntorch==2.4.0+cu124\\n    # via\\n    #   accelerate\\n    #   nvidia-modelopt\\n    #   optimum\\n    #   tensorrt-llm\\n    #   torchprofile\\n    #   torchvision\\ntorchprofile==0.1.0\\n    # via nvidia-modelopt\\ntorchvision==0.19.0+cu124\\n    # via\\n    #   nvidia-modelopt\\n    #   tensorrt-llm\\ntqdm==4.70.1\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   huggingface-hub\\n    #   nvidia-modelopt\\n    #   openai\\n    #   transformers\\ntransformers==4.45.1\\n    # via\\n    #   optimum\\n    #   tensorrt-llm\\ntriton==3.0.0\\n    # via torch\\ntyping-extensions==4.16.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\n    #   anyio\\n    #   exceptiongroup\\n    #   fastapi\\n    #   huggingface-hub\\n    #   multidict\\n    #   onnx\\n    #   openai\\n    #   pydantic\\n    #   pydantic-core\\n    #   torch\\n    #   typing-inspection\\n    #   uvicorn\\ntyping-inspection==0.4.4\\n    # via pydantic\\ntzdata==2026.5\\n    # via pandas\\nurllib3==2.8.0\\n    # via requests\\nuvicorn==0.54.0\\n    # via -r notebooks/trtllm-requirements.in\\nwheel==0.48.0\\n    # via tensorrt-llm\\nxxhash==4.0.1\\n    # via\\n    #   datasets\\n    #   evaluate\\nyarl==1.25.1\\n    # via aiohttp\\nzipp==4.1.1\\n    # via importlib-metadata\\n"], "notebooks/trtllm-build-constraints.txt": ["b398d2a7c7789034c0a99dc9cea8c8d60660994c6c3b13fc25d54b97f159cc1e", "# Build-time constraints for sdists in the TensorRT-LLM venv (uv pip install --build-constraints).\\n# mpi4py 3.1.6 builds from source and its conf/mpidistutils.py calls new_compiler(dry_run=...);\\n# setuptools 81.0.0 removed dry-run support (NEWS: \\"Removed support for the --dry-run parameter\\", #4872),\\n# so uv\'s isolated build env must use the last 80.x. Failure seen on Kaggle 2026-10-10 (v3b install-trtllm.log).\\nsetuptools==80.10.2\\n"], "inference/trtllm/build_engine.py": ["291c9cc5cbdcbabe9e71f5e27d696010ee1254e688bd6b0dde91a35a60d4e125", "\\"\\"\\"Build a TensorRT-LLM 0.14.0 FP16 engine for Qwen2.5-1.5B-Instruct on a Tesla T4 (SM75). Runs in the Kaggle\\nnotebook\'s Python 3.10 venv (notebooks/trtllm-requirements.txt).\\n\\n1. Untie the embeddings: copy the HF checkpoint with lm_head.weight = model.embed_tokens.weight and\\n   tie_word_embeddings false. TensorRT-LLM 0.13/0.14\'s Qwen converter fails on tied Qwen2 checkpoints\\n   (\\"\'NoneType\' object has no attribute \'to\'\\", NVIDIA/TensorRT-LLM issue 2388; Qwen2.5-1.5B support came in 0.15).\\n   The weights are the same tensors, so the model is unchanged.\\n2. Convert to a TensorRT-LLM checkpoint in FP16 with QWenForCausalLM.from_hugging_face (what\\n   examples/qwen/convert_checkpoint.py calls). The HF checkpoint is bf16; T4 has no bf16, as in the vLLM runs.\\n3. trtllm-build, as examples/qwen/README.md (FP16, --gemm_plugin float16) plus --context_fmha disable: the fused\\n   context attention kernels are not built for SM75 (fmhaRunner.cpp asserts \\"Unsupported architecture\\";\\n   NVIDIA/TensorRT-LLM issue 1515), so context attention runs unfused. The commands are written to --cmd-out.\\n\\nTwo engines from one checkpoint (PROFILES), because unfused context attention needs a plugin workspace that grows\\nwith max_batch_size x max_input_len^2. TensorRT-LLM 0.14\'s GPTAttentionPluginCommon::getWorkspaceSizeForContext\\n(gptAttentionCommon.cpp), FP16, 12 query heads, 2 KV heads, head size 128: the qk buffer (2 bytes) plus its float\\ncopy (4 bytes) take 6 x B x 12 x L^2 bytes, plus small terms. For B=32, L=3072, max_num_tokens 8192 it gives\\n21,978,227,456 bytes, exactly what the v3h build reported against 15,636,037,632 available\\n(scripts in the commit message; workspace_bytes() below). Hence:\\n- bench: B=32 (concurrency 32), L=1024 (longest loadgen prompt 893 tokens with the chat template), seq 1024+128:\\n  2.55 GB workspace.\\n- gsm8k: B=8, L=2048 (longest GSM8K 5-shot chat prompt 1654 tokens over the 250 items), seq 2048+256: 2.55 GB.\\n  Batch 8 only queues GSM8K requests (num_concurrent 16); greedy outputs do not depend on batching.\\nB=32 with L=2048 would need 9.85 GB, leaving too little of a 15 GB T4 for weights (about 3.1 GB) and KV cache.\\n\\npython build_engine.py HF_DIR WORK_DIR --cmd-out results/server_args/trtllm-build.txt\\n\\"\\"\\"\\n\\nimport argparse\\nimport json\\nimport shlex\\nimport shutil\\nimport subprocess\\nimport sys\\nimport time\\nfrom pathlib import Path\\n\\nBUILD_ARGS = [\\n    \\"--gemm_plugin\\", \\"float16\\",\\n    \\"--gpt_attention_plugin\\", \\"float16\\",\\n    \\"--context_fmha\\", \\"disable\\",\\n    \\"--remove_input_padding\\", \\"enable\\",\\n    \\"--kv_cache_type\\", \\"paged\\",\\n    \\"--max_num_tokens\\", \\"8192\\",\\n    \\"--workers\\", \\"1\\",\\n]  # fmt: skip\\n# name: (max_batch_size, max_input_len, max_seq_len)\\nPROFILES = {\\"bench\\": (32, 1024, 1024 + 128), \\"gsm8k\\": (8, 2048, 2048 + 256)}\\nHEADS, KV_HEADS, HEAD_SIZE = 12, 2, 128\\n\\n\\ndef workspace_bytes(batch, input_len, max_num_tokens=8192):\\n    \\"\\"\\"Unfused context attention workspace of TensorRT-LLM 0.14 (getWorkspaceSizeForContext), FP16, 256-byte aligned\\n    buffers. Reproduces the 21,978,227,456 bytes of the failed v3h build for (32, 3072, 8192).\\"\\"\\"\\n    s, b, length, t = 2, batch, input_len, max_num_tokens\\n    parts = [\\n        32 * 1024 * 1024,\\n        s * t * length,\\n        4 * (b + 1),\\n        4 * (b + 1),\\n        4 * (b + 1),\\n        4 * b * HEAD_SIZE // 2,\\n    ]\\n    parts += [\\n        s * t * HEADS * HEAD_SIZE,\\n        s * b * length * KV_HEADS * HEAD_SIZE,\\n        s * b * length * KV_HEADS * HEAD_SIZE,\\n    ]\\n    parts += [\\n        s * b * HEADS * length * length,\\n        s * t * HEADS * HEAD_SIZE,\\n        4 * b * HEADS * length * length,\\n        4 * t,\\n        4 * t,\\n    ]\\n    return sum((p + 255) // 256 * 256 for p in parts)\\n\\n\\ndef untie(src, dst):\\n    from safetensors.torch import load_file, save_file\\n\\n    dst.mkdir(parents=True, exist_ok=True)\\n    for f in src.iterdir():\\n        if f.is_file() and not f.name.startswith(\\"model\\") and f.name != \\"config.json\\":\\n            shutil.copy2(f, dst / f.name)\\n    tensors = {}\\n    for f in sorted(src.glob(\\"*.safetensors\\")):\\n        tensors.update(load_file(f))\\n    if \\"lm_head.weight\\" not in tensors:\\n        tensors[\\"lm_head.weight\\"] = tensors[\\"model.embed_tokens.weight\\"].clone()\\n    save_file(tensors, dst / \\"model.safetensors\\", metadata={\\"format\\": \\"pt\\"})\\n    cfg = json.loads((src / \\"config.json\\").read_text())\\n    cfg[\\"tie_word_embeddings\\"] = False\\n    (dst / \\"config.json\\").write_text(json.dumps(cfg, indent=2))\\n\\n\\ndef convert(untied, ckpt):\\n    \\"\\"\\"FP16 TensorRT-LLM checkpoint from the untied HF dir. 0.14\'s save_checkpoint writes into output_dir without\\n    creating it (failed on Kaggle 2026-10-10, v3f: \\"No such file or directory ... /tmp/trtllm/ckpt-fp16/.tmp...\\"),\\n    so the dir is created here. The model is deleted inside this function so PretrainedModel.__del__ (which calls\\n    torch.cuda.ipc_collect) runs while torch is intact, not at interpreter exit.\\"\\"\\"\\n    import gc\\n\\n    from tensorrt_llm.models import QWenForCausalLM\\n\\n    ckpt.mkdir(parents=True, exist_ok=True)\\n    model = QWenForCausalLM.from_hugging_face(str(untied), dtype=\\"float16\\")\\n    model.save_checkpoint(str(ckpt), save_config=True)\\n    del model\\n    gc.collect()\\n    return sorted(f.name for f in ckpt.iterdir())\\n\\n\\ndef build(ckpt, engine, cmd_out, profile):\\n    batch, input_len, seq_len = PROFILES[profile]\\n    engine.mkdir(parents=True, exist_ok=True)\\n    cmd = [\\n        str(Path(sys.executable).parent / \\"trtllm-build\\"),\\n        \\"--checkpoint_dir\\",\\n        str(ckpt),\\n        \\"--output_dir\\",\\n        str(engine),\\n        *BUILD_ARGS,\\n        \\"--max_batch_size\\",\\n        str(batch),\\n        \\"--max_input_len\\",\\n        str(input_len),\\n        \\"--max_seq_len\\",\\n        str(seq_len),\\n    ]\\n    Path(cmd_out).parent.mkdir(parents=True, exist_ok=True)\\n    with open(cmd_out, \\"a\\") as f:\\n        f.write(\\n            f\\"# {profile}: attention workspace {workspace_bytes(batch, input_len)} bytes\\\\n{shlex.join(cmd)}\\\\n\\"\\n        )\\n    subprocess.run(cmd, check=True)\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(\\n        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter\\n    )\\n    p.add_argument(\\"hf_dir\\")\\n    p.add_argument(\\"work_dir\\")\\n    p.add_argument(\\"--cmd-out\\", required=True)\\n    args = p.parse_args()\\n    hf, work = Path(args.hf_dir), Path(args.work_dir)\\n    untied, ckpt = work / \\"hf-untied\\", work / \\"ckpt-fp16\\"\\n\\n    t = time.time()\\n    untie(hf, untied)\\n    print(f\\"untied checkpoint in {time.time() - t:.0f}s\\", flush=True)\\n    t = time.time()\\n    files = convert(untied, ckpt)\\n    print(f\\"converted checkpoint in {time.time() - t:.0f}s: {files}\\", flush=True)\\n    Path(args.cmd_out).unlink(missing_ok=True)\\n    for profile in PROFILES:\\n        t = time.time()\\n        build(ckpt, work / f\\"engine-fp16-{profile}\\", args.cmd_out, profile)\\n        print(f\\"built {profile} engine in {time.time() - t:.0f}s\\", flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"], "inference/trtllm/serve.py": ["96928b90b0e8dfd1a70ceabd38f45ca5443645a6a1bf17c01be1bb5b114fe805", "\\"\\"\\"OpenAI-compatible server for a prebuilt TensorRT-LLM 0.14.0 engine, with Prometheus metrics at /metrics.\\n\\nTensorRT-LLM 0.14.0 is the last release whose support matrix lists Turing (SM75). It has no trtllm-serve (added in\\n0.15.0); its OpenAI server is examples/apps/openai_server.py in the source tree. The notebook downloads that file at\\ntag v0.14.0 (sha256 checked); this launcher imports a copy with one patch (4.) and adds:\\n1. ignore_eos: 0.14\'s request models accept ignore_eos but to_sampling_params() drops it, and SamplingParams has no\\n   ignore_eos until 0.15. When a request sets it, end_id and pad_id become -1, which is what 0.14\'s own trtllm-bench\\n   does (tensorrt_llm/bench/benchmark/throughput.py) to always decode max_tokens.\\n2. /metrics (prometheus_client): HTTP requests by path and status, and prompt tokens submitted. 0.14\'s LLM API\\n   exposes no iteration or KV cache stats, so there are no engine-side metrics.\\n3. The upstream entrypoint builds an engine on the fly with max_batch_size 10; this loads the prebuilt engine.\\n4. Streaming: upstream chat_stream_generator marks a choice finished on the first update whose text diff is empty\\n   (`else: finish_reason_sent[i] = True`), so an empty diff mid-stream (e.g. a token that decodes to nothing yet)\\n   silences the rest of the stream while usage still reports every token, and client TPOT/ITL undercount.\\n   STREAM_PATCH makes that branch `elif output.finish_reason is not None:`, so only a finished sequence ends the\\n   stream. The load generator still flags truncated streams (benchmark/loadgen/metrics.py stream_truncated).\\n5. Detokenization skips special tokens (0.14\'s TransformersTokenizer.decode passes no skip_special_tokens, so a\\n   trailing <|im_end|> could reach the text). vLLM\'s default is also skip_special_tokens=True; affects GSM8K text.\\nKV cache block reuse (prefix caching) is off, as --no-enable-prefix-caching on vLLM. Temperature 0 needs no patch:\\nthe 0.14 runtime maps it to greedy (top_k 1) in runtime/samplingConfig.h.\\n\\npython serve.py ENGINE_DIR --tokenizer HF_DIR --served-model-name Qwen/Qwen2.5-1.5B-Instruct \\\\\\n  --openai-server-dir DIR_WITH_openai_server.py --port 8000 --kv-cache-free-gpu-memory-fraction 0.9\\n\\"\\"\\"\\n\\nimport argparse\\nimport asyncio\\nimport importlib.util\\nfrom pathlib import Path\\n\\nINDENT = \\" \\" * 20\\nSTREAM_PATCH = (\\n    f\\"{INDENT}else:\\\\n{INDENT}    finish_reason_sent[i] = True\\\\n\\",\\n    f\\"{INDENT}elif output.finish_reason is not None:\\\\n{INDENT}    finish_reason_sent[i] = True\\\\n\\",\\n)\\n\\n\\ndef patched_server_source(src):\\n    \\"\\"\\"Upstream openai_server.py source with STREAM_PATCH applied; raises if the snippet is not there exactly once.\\"\\"\\"\\n    old, new = STREAM_PATCH\\n    if src.count(old) != 1:\\n        raise ValueError(\\n            f\\"stream patch target found {src.count(old)} times, expected 1\\"\\n        )\\n    return src.replace(old, new)\\n\\n\\ndef load_server_module(server_dir):\\n    path = Path(server_dir) / \\"openai_server_patched.py\\"\\n    path.write_text(\\n        patched_server_source((Path(server_dir) / \\"openai_server.py\\").read_text())\\n    )\\n    spec = importlib.util.spec_from_file_location(\\"openai_server_patched\\", path)\\n    module = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(module)\\n    return module\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(\\n        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter\\n    )\\n    p.add_argument(\\"engine_dir\\")\\n    p.add_argument(\\n        \\"--tokenizer\\", required=True, help=\\"HF model dir (tokenizer and chat template)\\"\\n    )\\n    p.add_argument(\\"--served-model-name\\", required=True)\\n    p.add_argument(\\n        \\"--openai-server-dir\\",\\n        required=True,\\n        help=\\"directory holding v0.14.0 openai_server.py\\",\\n    )\\n    p.add_argument(\\"--host\\", default=\\"127.0.0.1\\")\\n    p.add_argument(\\"--port\\", type=int, default=8000)\\n    p.add_argument(\\"--kv-cache-free-gpu-memory-fraction\\", type=float, default=0.9)\\n    args = p.parse_args()\\n\\n    openai_server = load_server_module(args.openai_server_dir)  # v0.14.0 + STREAM_PATCH\\n    from fastapi.responses import Response\\n    from prometheus_client import CONTENT_TYPE_LATEST, Counter, generate_latest\\n    from tensorrt_llm.hlapi import LLM, KvCacheConfig\\n    from tensorrt_llm.hlapi.openai_protocol import (\\n        ChatCompletionRequest,\\n        CompletionRequest,\\n    )\\n    from tensorrt_llm.hlapi.tokenizer import TransformersTokenizer\\n    from transformers import AutoTokenizer\\n\\n    for cls in (ChatCompletionRequest, CompletionRequest):\\n\\n        def to_sampling_params(self, _orig=cls.to_sampling_params):\\n            sp = _orig(self)\\n            if self.ignore_eos:\\n                sp.end_id, sp.pad_id = -1, -1\\n            return sp\\n\\n        cls.to_sampling_params = to_sampling_params\\n\\n    decode = TransformersTokenizer.decode\\n\\n    def decode_skip_special(self, token_ids, *a, **kw):\\n        return decode(self, token_ids, *a, **{\\"skip_special_tokens\\": True, **kw})\\n\\n    TransformersTokenizer.decode = decode_skip_special\\n\\n    llm = LLM(\\n        args.engine_dir,\\n        tokenizer=args.tokenizer,\\n        kv_cache_config=KvCacheConfig(\\n            free_gpu_memory_fraction=args.kv_cache_free_gpu_memory_fraction,\\n            enable_block_reuse=False,\\n        ),\\n    )\\n    requests = Counter(\\n        \\"trtllm_http_requests\\", \\"HTTP requests by path and status\\", [\\"path\\", \\"status\\"]\\n    )\\n    prompt_tokens = Counter(\\n        \\"trtllm_prompt_tokens\\", \\"Prompt tokens submitted to the engine\\"\\n    )\\n    generate_async = llm.generate_async\\n\\n    def counted_generate_async(*a, **kw):\\n        promise = generate_async(*a, **kw)\\n        prompt_tokens.inc(len(promise.prompt_token_ids))\\n        return promise\\n\\n    llm.generate_async = counted_generate_async\\n    server = openai_server.OpenaiServer(\\n        llm=llm,\\n        model=args.served_model_name,\\n        hf_tokenizer=AutoTokenizer.from_pretrained(args.tokenizer),\\n    )\\n\\n    @server.app.middleware(\\"http\\")\\n    async def count_requests(request, call_next):\\n        resp = await call_next(request)\\n        requests.labels(request.url.path, str(resp.status_code)).inc()\\n        return resp\\n\\n    server.app.add_api_route(\\n        \\"/metrics\\",\\n        lambda: Response(generate_latest(), media_type=CONTENT_TYPE_LATEST),\\n        methods=[\\"GET\\"],\\n    )\\n    asyncio.run(server(args.host, args.port))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"]}'
)
for rel, (sha, text) in FILES.items():
    assert hashlib.sha256(text.encode()).hexdigest() == sha, rel
    p = pathlib.Path(BENCH) / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
    if rel.endswith(".sh"):
        os.chmod(p, 0o755)
print("wrote", len(FILES), "files to", BENCH)

In [ ]:
# Hardware and base image capture, before installing anything.
import json
import os
import pathlib
import shlex
import subprocess
import sys
import time

for d in ("env", "logs", "raw", "server_args"):
    pathlib.Path(RESULTS, d).mkdir(parents=True, exist_ok=True)


def sh(cmd, out=None, check=False, timeout=None):
    """Run a shell command, save stdout+stderr to RESULTS/out, raise on non-zero exit if check. With timeout (s),
    coreutils timeout kills the command's whole process group, so a hang ends in exit 124 instead of the session."""
    if timeout:
        cmd = f"timeout -k 60 {int(timeout)} bash -c {shlex.quote(cmd)}"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=False)
    text = r.stdout + r.stderr
    if out:
        pathlib.Path(RESULTS, out).write_text(text)
    if check and r.returncode != 0:
        why = (
            f" (timed out after {timeout} s)" if timeout and r.returncode == 124 else ""
        )
        raise RuntimeError(f"exit {r.returncode}{why}: {cmd[:300]}\n{text[-3000:]}")
    return text


sh("nvidia-smi -q", "env/nvidia-smi-q.txt")
print(sh("nvidia-smi", "env/nvidia-smi.txt"))
GPU_COUNT = len(
    [ln for ln in sh("nvidia-smi -L").splitlines() if ln.startswith("GPU ")]
)
sh("lscpu", "env/lscpu.txt")
sh("free -m", "env/free.txt")
sh("cat /etc/os-release", "env/os-release.txt")
sh(f"{sys.executable} -m pip freeze", "env/kaggle-base-pip-freeze.txt")
pathlib.Path(RESULTS, "env/run.json").write_text(
    json.dumps(
        {
            "git_commit": GIT_COMMIT,
            "gpu_count": GPU_COUNT,
            **PINS,
            "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "kaggle_docker_image": os.environ.get("KAGGLE_DOCKER_IMAGE"),
            "kaggle_kernel_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
        },
        indent=1,
    )
)
print("GPU_COUNT =", GPU_COUNT)

In [ ]:
# Preflight: fail fast instead of burning the session. Run v1 (2026-10-07) had no GPU and no internet.
import shutil
import urllib.request

problems = []
if not shutil.which("nvidia-smi"):
    problems.append("nvidia-smi not found")
elif GPU_COUNT == 0:
    problems.append("nvidia-smi reports 0 GPUs")
for url in PREFLIGHT_URLS:
    try:
        urllib.request.urlopen(url, timeout=10)
    except OSError as e:
        problems.append(f"{url} not reachable: {e}")
if problems:
    msg = "no GPU/internet: check Kaggle phone verification and that the accelerator is GPU T4 x2 and Internet is on"
    pathlib.Path(RESULTS, "FAILED-preflight.txt").write_text(
        msg + "\n" + "\n".join(problems) + "\n"
    )
    raise RuntimeError(msg + " (" + "; ".join(problems) + ")")
print(f"preflight OK: {GPU_COUNT} GPU(s), reachable: {', '.join(PREFLIGHT_URLS)}")

In [ ]:
import signal
import traceback
import urllib.request


def wait_ready(proc, timeout=1200):
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            raise RuntimeError(f"server exited with code {proc.returncode}")
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5)
            return time.time() - t
        except OSError:
            time.sleep(5)
    raise TimeoutError("server not ready")


def start_server(cell, model_key, tp=1, extra=()):
    repo, rev = MODELS[model_key]
    args = [
        f"{VENVS}/vllm/bin/vllm",
        "serve",
        repo,
        "--revision",
        rev,
        "--tokenizer-revision",
        rev,
        "--served-model-name",
        repo,
        "--host",
        "127.0.0.1",
        "--port",
        str(PORT),
        "--dtype",
        "float16",
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEM_UTIL),
        "--attention-backend",
        "TRITON_ATTN",
        "--no-enable-prefix-caching",
        "--seed",
        "0",
        # vLLM defaults: without this, the model's generation_config.json (repetition_penalty 1.1) applies to
        # every request that does not set it (seen in run v2, 2026-10-09).
        "--generation-config",
        "vllm",
        "--tensor-parallel-size",
        str(tp),
        *extra,
    ]
    cmd = shlex.join(args)
    pathlib.Path(RESULTS, "server_args", f"{cell}.txt").write_text(cmd + "\n")
    log = open(pathlib.Path(RESULTS, "logs", f"server-{cell}.log"), "w")  # noqa: SIM115 open for the server's lifetime
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=",".join(str(i) for i in range(tp)))
    proc = subprocess.Popen(
        args, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True
    )
    try:
        print(f"{cell}: server ready in {wait_ready(proc):.0f}s")
    except Exception:
        stop_server(proc, cell)
        raise
    return proc, cmd


def gpu_mem_used():
    out = sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits")
    return [int(x) for x in out.split() if x.isdigit()]


def stop_server(proc, cell, timeout=180):
    """SIGTERM the server's process group, then SIGKILL it (EngineCore and TP workers live in the same group),
    then wait until every GPU is under 1 GiB used so the next cell starts on an empty GPU."""
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(120)
    except (ProcessLookupError, subprocess.TimeoutExpired):
        pass
    try:
        os.killpg(proc.pid, signal.SIGKILL)  # leftover workers, if any
    except ProcessLookupError:
        pass
    t, used = time.time(), gpu_mem_used()
    while any(u >= 1024 for u in used) and time.time() - t < timeout:
        time.sleep(5)
        used = gpu_mem_used()
    msg = f"{cell}: GPU memory.used MiB after stop {used} ({time.time() - t:.0f}s)"
    pathlib.Path(RESULTS, "logs", f"gpu-drain-{cell}.txt").write_text(msg + "\n")
    print(msg if all(u < 1024 for u in used) else "WARNING " + msg)


def loadgen(
    cell,
    model_key,
    precision,
    concurrency,
    cmd,
    gpus,
    extra="",
    env_prefix="",
    engine="vllm",
    engine_version=None,
):
    repo, rev = MODELS[model_key]
    c = " ".join(str(x) for x in concurrency)
    gpu_ids = ",".join(
        str(i) for i in range(gpus)
    )  # nvidia-smi sampler watches only the server's GPUs
    out = sh(
        f"{env_prefix} CUDA_VISIBLE_DEVICES={gpu_ids} {VENVS}/bench/bin/python {BENCH}/benchmark/loadgen/run.py --base-url http://127.0.0.1:{PORT}/v1 "
        f"--model {repo} --model-revision {rev} --engine {engine} --engine-version {engine_version or VLLM_VERSION} --precision {precision} "
        f"--label {cell} --hardware '{gpus}x Tesla T4 (Kaggle)' --concurrency {c} --repeats {REPEATS} "
        f"--metrics-url http://127.0.0.1:{PORT}/metrics --server-args {shlex.quote(cmd)} --git-commit {GIT_COMMIT} "
        f"--out-dir {RESULTS}/raw/{cell} {extra}",
        f"logs/loadgen-{cell}.log",
        check=True,
        timeout=3600,
    )
    print(out[-3000:])


def gsm8k(label, model_key):
    repo, _ = MODELS[model_key]
    out = sh(
        f"PATH={VENVS}/bench/bin:$PATH bash {BENCH}/benchmark/accuracy/run_gsm8k.sh "
        f"http://127.0.0.1:{PORT}/v1/chat/completions {repo} {RESULTS}/raw/accuracy/{label} {GSM8K_LIMIT} 16",
        check=True,
        timeout=3600,
    )
    print(out[-1500:])


def run_cell(
    cell,
    model_key,
    precision,
    concurrency,
    tp=1,
    accuracy_label=None,
    loadgen_extra="",
    loadgen_env="",
):
    t, proc = time.time(), None
    try:
        proc, cmd = start_server(cell, model_key, tp)
        loadgen(
            cell,
            model_key,
            precision,
            concurrency,
            cmd,
            tp,
            extra=loadgen_extra,
            env_prefix=loadgen_env,
        )
        if accuracy_label:
            gsm8k(accuracy_label, model_key)
    except Exception:  # noqa: BLE001 record and move on to the next cell
        pathlib.Path(RESULTS, f"FAILED-{cell}.txt").write_text(traceback.format_exc())
        print(f"{cell} FAILED, see results/FAILED-{cell}.txt")
    finally:
        if proc:
            stop_server(proc, cell)
    pathlib.Path(RESULTS, "env", f"timing-{cell}.txt").write_text(
        f"{time.time() - t:.0f} s wall clock\n"
    )
    print(f"{cell}: {time.time() - t:.0f}s")

In [ ]:
# Step runner. Every unit of work is one step(): it is skipped with NOT_RUN-<name>.txt when the remaining session
# budget cannot cover its estimate, records FAILED-<name>.txt with the traceback on any exception (the notebook
# moves on), and writes env/timing-<name>.txt. Returns fn's value (True for None) on success, None otherwise.
import hashlib
import urllib.request

T0 = time.time()


def time_left():
    return BUDGET_S - (time.time() - T0)


def not_run(name, reason):
    pathlib.Path(RESULTS, f"NOT_RUN-{name}.txt").write_text(f"not run: {reason}\n")
    print(name, "not run:", reason)


def step(name, est_s, fn, *args, **kw):
    if time_left() < est_s:
        not_run(
            name,
            f"time budget: {time_left():.0f} s left of {BUDGET_S} s, step estimate {est_s} s",
        )
        return None
    t = time.time()
    try:
        res = fn(*args, **kw)
        return True if res is None else res
    except Exception:  # noqa: BLE001 record and move on
        pathlib.Path(RESULTS, f"FAILED-{name}.txt").write_text(traceback.format_exc())
        print(f"{name} FAILED, see results/FAILED-{name}.txt")
        return None
    finally:
        pathlib.Path(RESULTS, "env", f"timing-{name}.txt").write_text(
            f"{time.time() - t:.0f} s wall clock\n"
        )
        print(f"{name}: {time.time() - t:.0f}s")


def fetch(url, sha, dest):
    """Download url to dest and check its sha256 (raises on mismatch)."""
    dest = pathlib.Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url, timeout=300) as r:
        data = r.read()
    got = hashlib.sha256(data).hexdigest()
    if got != sha:
        raise ValueError(f"{url}: sha256 {got} != pinned {sha}")
    dest.write_bytes(data)
    return dest


def install_vllm():
    """vLLM 0.31.0 venv from the run v2 lock (notebooks/vllm-requirements.txt)."""
    log = sh(
        f"uv venv -q --python {PYTHON_VERSION} {VENVS}/vllm && uv pip install -q --python {VENVS}/vllm/bin/python "
        f"--torch-backend cu129 -r {BENCH}/notebooks/vllm-requirements.txt",
        "logs/install-vllm.log",
        check=True,
        timeout=2400,
    )
    print(log[-1500:])
    sh(f"uv pip freeze --python {VENVS}/vllm/bin/python", "env/pip-freeze-vllm.txt")


def start_proc(cell, args, env, ready_url, timeout=900):
    """Start a server in its own process group, log to logs/server-<cell>.log, wait until ready_url answers 200."""
    cmd = shlex.join(args)
    pathlib.Path(RESULTS, "server_args", f"{cell}.txt").write_text(cmd + "\n")
    log = open(pathlib.Path(RESULTS, "logs", f"server-{cell}.log"), "w")  # noqa: SIM115 open for the server's lifetime
    proc = subprocess.Popen(
        args, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True
    )
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            raise RuntimeError(
                f"{cell}: server exited with code {proc.returncode}, see logs/server-{cell}.log"
            )
        try:
            urllib.request.urlopen(ready_url, timeout=5)
            print(f"{cell}: ready in {time.time() - t:.0f}s")
            return proc, cmd
        except OSError:
            time.sleep(5)
    stop_server(proc, cell)
    raise TimeoutError(f"{cell}: not ready after {timeout}s")

## Install the client venv (estimate 2 min)

In [ ]:
# uv and the client venv (loadgen, COCO fetch, lm-eval). Other venvs are installed by the steps that need them.
def install_base():
    print(
        sh(
            f"pip install -q uv=={UV_VERSION} && uv --version",
            "logs/install-uv.log",
            check=True,
            timeout=600,
        )
    )
    sh(
        f"uv venv -q --python {PYTHON_VERSION} {VENVS}/bench && "
        f"uv pip install -q --python {VENVS}/bench/bin/python -r {BENCH}/benchmark/requirements.txt",
        "logs/install-bench.log",
        check=True,
        timeout=1200,
    )
    sh(f"uv pip freeze --python {VENVS}/bench/bin/python", "env/pip-freeze-bench.txt")


BASE_OK = step("install-base", 600, install_base)

## Same-session vLLM 0.31.0 control, 1.5B FP16 c1/c8/c32 + GSM8K (estimate 35 min)

In [ ]:
# Same-session vLLM control: run v2's cell with --generation-config vllm (start_server), same loadgen and GSM8K.
# Runs before TensorRT-LLM so a slow TensorRT-LLM step cannot squeeze it out of the budget. Needs the base install.
if not RUN_VLLM_CONTROL:
    not_run(
        VLLM_CELL, f"RUN_VLLM_CONTROL is False: already run, see {CONTROL_EVIDENCE}"
    )
elif not pathlib.Path(VENVS, "bench/bin/python").exists():
    not_run(VLLM_CELL, "client venv missing (the base install step failed)")
elif step("install-vllm", 1500, install_vllm):
    step(
        VLLM_CELL,
        1800,
        run_cell,
        VLLM_CELL,
        "1.5b-fp16",
        "fp16",
        [1, 8, 32],
        accuracy_label="gsm8k-vllm-1.5b-fp16",
    )

## TensorRT-LLM 0.14.0: install, build, serve, loadgen, GSM8K (estimate 80 min)

In [ ]:
# TensorRT-LLM: install, download, build, serve, benchmark. Each step needs the previous one.
TRT_ENV = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="0",
    OMPI_ALLOW_RUN_AS_ROOT="1",
    OMPI_ALLOW_RUN_AS_ROOT_CONFIRM="1",
)
TPY = f"{VENVS}/trtllm/bin/python"


def install():
    # uv and the client venv come from the base install cell, which runs before the vLLM control
    print(
        sh(
            f"apt-get update -qq && apt-get install -y -qq --no-install-recommends openmpi-bin={OPENMPI_APT} "
            f"libopenmpi-dev={OPENMPI_APT} && mpirun --version",
            "logs/install-openmpi.log",
            check=True,
            timeout=900,
        )[-1000:]
    )
    # about 9 GB of wheels (tensorrt_llm 1.9 GB, TensorRT 10.4 libs, torch cu124, CUDA libs); 12 GB installed
    print(
        sh(
            f"uv python install {TRTLLM_PYTHON} && uv venv -q --python {TRTLLM_PYTHON} {VENVS}/trtllm && "
            f"UV_HTTP_TIMEOUT=300 uv pip install -q --python {TPY} --torch-backend cu124 --extra-index-url https://pypi.nvidia.com "
            f"--index-strategy unsafe-best-match --build-constraints {BENCH}/notebooks/trtllm-build-constraints.txt "
            f"-r {BENCH}/notebooks/trtllm-requirements.txt",
            "logs/install-trtllm.log",
            check=True,
            timeout=3600,
        )[-1500:]
    )
    sh(f"uv pip freeze --python {TPY}", "env/pip-freeze-trtllm.txt")
    # dependency consistency, logged even when it passes (a pyarrow/datasets mismatch broke the import in v3d)
    print(sh(f"uv pip check --python {TPY}; echo exit=$?", "env/trtllm-pip-check.txt"))
    fetch(
        OPENAI_SERVER_URL, OPENAI_SERVER_SHA256, "/tmp/trtllm-example/openai_server.py"
    )
    print(
        sh(
            f"cd /tmp && {TPY} -c 'import torch, tensorrt, tensorrt_llm; print(tensorrt_llm.__version__, tensorrt.__version__, "
            "torch.__version__, torch.version.cuda, torch.cuda.get_device_capability(0))'",
            "env/trtllm-versions.txt",
            check=True,
        )
    )


def download():
    repo, rev = MODELS["1.5b-fp16"]
    print(
        sh(
            f'{TPY} -c "from huggingface_hub import snapshot_download; '
            f"print(snapshot_download('{repo}', revision='{rev}', local_dir='{TRT_WORK}/hf'))\"",
            "logs/hf-download.log",
            check=True,
            timeout=1200,
        )[-500:]
    )


def build_engine():
    env = " ".join(
        f"{k}={TRT_ENV[k]}"
        for k in (
            "CUDA_VISIBLE_DEVICES",
            "OMPI_ALLOW_RUN_AS_ROOT",
            "OMPI_ALLOW_RUN_AS_ROOT_CONFIRM",
        )
    )
    out = sh(
        f"{env} {TPY} {BENCH}/inference/trtllm/build_engine.py {TRT_WORK}/hf {TRT_WORK} "
        f"--cmd-out {RESULTS}/server_args/trtllm-build.txt",
        "logs/trtllm-build.log",
        check=True,
        timeout=3600,
    )
    print(out[-3000:])
    for p in ("bench", "gsm8k"):
        sh(
            f"cp {TRT_WORK}/engine-fp16-{p}/config.json {RESULTS}/env/trtllm-engine-config-{p}.json && "
            f"du -sh {TRT_WORK}/engine-fp16-{p}",
            f"env/trtllm-engine-size-{p}.txt",
            check=True,
        )


def serve(name, profile):
    """Start serve.py on one engine profile (build_engine.PROFILES): bench (batch 32, input 1024) for loadgen,
    gsm8k (batch 8, input 2048) for GSM8K. One engine cannot do both on a T4 (attention workspace, see build_engine)."""
    args = [
        TPY,
        f"{BENCH}/inference/trtllm/serve.py",
        f"{TRT_WORK}/engine-fp16-{profile}",
        "--tokenizer",
        f"{TRT_WORK}/hf",
        "--served-model-name",
        MODELS["1.5b-fp16"][0],
        "--openai-server-dir",
        "/tmp/trtllm-example",
        "--port",
        str(PORT),
        "--kv-cache-free-gpu-memory-fraction",
        str(KV_FRACTION),
    ]
    return start_proc(
        name, args, TRT_ENV, f"http://127.0.0.1:{PORT}/health", timeout=900
    )


def serve_and_bench():
    proc, cmd = serve(CELL, "bench")
    try:
        step(
            f"{CELL}-loadgen",
            1800,
            loadgen,
            CELL,
            "1.5b-fp16",
            "fp16",
            [1, 8, 32],
            cmd,
            1,
            engine="trtllm",
            engine_version=TRTLLM_VERSION,
        )
    finally:
        stop_server(proc, CELL)


def serve_and_gsm8k():
    proc, _ = serve("trtllm-t4-1.5b-fp16-gsm8k", "gsm8k")
    try:
        gsm8k("gsm8k-trtllm-1.5b-fp16", "1.5b-fp16")
    finally:
        stop_server(proc, "trtllm-t4-1.5b-fp16-gsm8k")


not_run(
    "triton-tensorrtllm-backend",
    "Triton's TensorRT-LLM backend ships only in the NGC "
    "tritonserver:<ver>-trtllm-python-py3 container image, which needs Docker; Kaggle has no Docker. "
    "TensorRT-LLM is served with its own OpenAI server instead (inference/trtllm/serve.py)",
)
ok = True
for name, est, fn in (
    ("install", 1800, install),
    ("download", 600, download),
    ("build-engine", 1800, build_engine),
    (CELL, 2700, serve_and_bench),
    ("gsm8k-trtllm-1.5b-fp16", 1200, serve_and_gsm8k),
):
    if not ok:
        not_run(
            name,
            "an earlier TensorRT-LLM step failed or was skipped (see FAILED-*/NOT_RUN-*)",
        )
        continue
    ok = bool(step(name, est, fn))

## Summary

In [ ]:
# Quick look. The real report is benchmark/report.py, run offline after downloading the results.
# Wrapped like a step: nothing in the summary may raise, so the end-of-session files are always written.
import glob


def summary():
    for f in sorted(glob.glob(f"{RESULTS}/raw/**/*.json", recursive=True)):
        try:
            d = json.loads(pathlib.Path(f).read_text())
            print(
                os.path.relpath(f, RESULTS),
                json.dumps(d.get("summary") or d.get("metrics"))[:300],
            )
        except (OSError, ValueError, AttributeError) as e:
            print(f, "unreadable:", e)
    print(
        sh(f"ls {RESULTS}/FAILED-* {RESULTS}/NOT_RUN-* 2>/dev/null")
        or "no failed or skipped steps"
    )


step("summary", 0, summary)
sh(
    "nvidia-smi -q", "env/nvidia-smi-q-end.txt"
)  # cumulative clock event reason counters over the session
pathlib.Path(RESULTS, "env/finished_utc.txt").write_text(
    time.strftime("%Y-%m-%dT%H:%M:%SZ\n", time.gmtime())
)
print(f"session used {time.time() - T0:.0f} s of the {BUDGET_S} s budget")